# RAMP++-NCL-MFB TC5 End-to-End Notebook

This notebook implements the team-rule-aligned TC5 `RAMP++-NCL-MFB` pipeline for the five shared datasets.

It uses the pinned TabM paper templates, canonical split-default data, paper-style reporting (`0.toml` + `report.json` + `DONE`), and reports mean / best-head / greedy-heads inference modes for every run.


In [ ]:
# Optional dependency installation.
import os
import sys
import subprocess
from pathlib import Path

INSTALL_REQUIREMENTS = os.environ.get('INSTALL_REQUIREMENTS_IN_NOTEBOOK', '0').strip().lower() in {'1', 'true', 'yes', 'on'}
REQUIREMENTS_FILE = Path(os.environ.get('REQUIREMENTS_FILE', 'requirements.txt')).expanduser().resolve()

if INSTALL_REQUIREMENTS:
    if not REQUIREMENTS_FILE.exists():
        raise FileNotFoundError(f'requirements file not found: {REQUIREMENTS_FILE}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REQUIREMENTS_FILE)], check=True)
    print(f'Installed requirements from {REQUIREMENTS_FILE}')
else:
    print('Skipping inline dependency installation. Set INSTALL_REQUIREMENTS_IN_NOTEBOOK=1 to enable it.')

# Standard library imports.
import gc
import hashlib
import itertools
import json
import math
import random
import shutil
import time
import tomllib
from copy import deepcopy
from dataclasses import dataclass
from datetime import timedelta
from typing import Any

import matplotlib

matplotlib.use(os.environ.get('MPLBACKEND', 'Agg'))

import matplotlib.pyplot as plt
import delu
import numpy as np
import pandas as pd
import rtdl_num_embeddings
import rtdl_revisiting_models
import sklearn.metrics as skm
import sklearn.preprocessing as skp
import tomli_w
from scipy.optimize import minimize

TABM_IMPORT_ROOTS = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve() / 'tabm',
    Path.cwd().resolve().parent / 'tabm',
]
for root in TABM_IMPORT_ROOTS:
    if root.exists() and (root.name == 'tabm' or root.joinpath('tabm').exists()):
        root_str = str(root)
        if root_str not in sys.path:
            sys.path.insert(0, root_str)

import tabm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


try:
    from IPython.display import display
except Exception:
    def display(obj: Any) -> None:
        print(obj)

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


def env_str(name: str, default: str) -> str:
    value = os.environ.get(name)
    return default if value is None or value.strip() == '' else value.strip()


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    value = value.strip().lower()
    if value in {'1', 'true', 'yes', 'on'}:
        return True
    if value in {'0', 'false', 'no', 'off'}:
        return False
    raise ValueError(f'Invalid boolean for {name}: {value}')


def env_int(name: str, default: int) -> int:
    value = os.environ.get(name)
    return default if value is None or value.strip() == '' else int(value)


def env_float(name: str, default: float) -> float:
    value = os.environ.get(name)
    return default if value is None or value.strip() == '' else float(value)


def env_csv(name: str, default: list[str]) -> list[str]:
    value = os.environ.get(name)
    if value is None or value.strip() == '':
        return list(default)
    return [x.strip() for x in value.split(',') if x.strip()]


def env_int_list(name: str, default: list[int]) -> list[int]:
    value = os.environ.get(name)
    if value is None or value.strip() == '':
        return list(default)
    return [int(x.strip()) for x in value.split(',') if x.strip()]


def unique_paths(paths: list[Path]) -> list[Path]:
    result = []
    seen = set()
    for path in paths:
        try:
            resolved = path.expanduser().resolve()
        except FileNotFoundError:
            resolved = path.expanduser().absolute()
        key = str(resolved)
        if key not in seen:
            result.append(resolved)
            seen.add(key)
    return result


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')


def stable_int_from_parts(*parts: Any) -> int:
    text = '||'.join(map(str, parts))
    return int(hashlib.md5(text.encode('utf-8')).hexdigest()[:8], 16)


PROJECT_ROOT = Path(env_str('PROJECT_ROOT', str(Path.cwd().resolve()))).expanduser().resolve()
WORK_ROOT = Path(env_str('WORK_ROOT', str(PROJECT_ROOT))).expanduser().resolve()
ARTIFACT_ROOT = Path(env_str('ARTIFACT_ROOT', str(WORK_ROOT / 'artifacts' / 'ramp_ncl_mfb_tc5'))).expanduser().resolve()
RAW_RESULT_ROOT = ARTIFACT_ROOT / 'raw'
IMPORTED_RESULT_ROOT = ARTIFACT_ROOT / 'imported_candidates'
TABLE_ROOT = ARTIFACT_ROOT / 'tables'
PLOT_ROOT = ARTIFACT_ROOT / 'plots'
CACHE_ROOT = ARTIFACT_ROOT / 'cache'
TEAM_EXPERIMENT_ROOT = ARTIFACT_ROOT / 'paper' / 'exp' / 'ramp_ncl_mfb_tc5'
TEAM_AGGREGATED_ROOT = TEAM_EXPERIMENT_ROOT / '_aggregated'
for path in [ARTIFACT_ROOT, RAW_RESULT_ROOT, IMPORTED_RESULT_ROOT, TABLE_ROOT, PLOT_ROOT, CACHE_ROOT, TEAM_EXPERIMENT_ROOT, TEAM_AGGREGATED_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

TEAM_DATASETS = ['homesite', 'sberbank', 'ecom-offers', 'cooking-time', 'delivery-eta']
SCREEN_SEED = env_int('SCREEN_SEED', 0)
FINAL_SEEDS = env_int_list('FINAL_SEEDS', [0, 1, 2])
SPLIT_NAME = env_str('SPLIT_NAME', 'default')
NUM_WORKERS = env_int('NUM_WORKERS', 2)
EVAL_BATCH_SIZE = env_int('EVAL_BATCH_SIZE', 8192)
MAX_EPOCHS = env_int('MAX_EPOCHS', 200)
USE_AMP = env_bool('USE_AMP', True)
FORCE_RERUN = env_bool('FORCE_RERUN', False)
SMOKE = env_bool('SMOKE', False)
DEBUG_MAX_TRAIN_ROWS = env_int('DEBUG_MAX_TRAIN_ROWS', 0)
DEBUG_MAX_VAL_ROWS = env_int('DEBUG_MAX_VAL_ROWS', 0)
DEBUG_MAX_TEST_ROWS = env_int('DEBUG_MAX_TEST_ROWS', 0)
FEATURE_SCORE_BATCHES = env_int('FEATURE_SCORE_BATCHES', 8)
FEATURE_SCORE_MAX_SAMPLES = env_int('FEATURE_SCORE_MAX_SAMPLES', 16384)
ROUTER_ENABLED_DEFAULT = env_bool('ROUTER_ENABLED_DEFAULT', False)
ENABLE_ROUTER_SCREEN = env_bool('ENABLE_ROUTER_SCREEN', True)
BLEND_REGULARIZATION_GRID = [float(x) for x in env_csv('BLEND_REGULARIZATION_GRID', ['0.0', '1e-4', '1e-3', '1e-2'])]
CLASSIFICATION_BLEND_METHODS = env_csv('CLASSIFICATION_BLEND_METHODS', ['logloss', 'auc_surrogate'])

candidate_previous_roots = [
    WORK_ROOT.parent / 'mfb_tabm_tc2',
    WORK_ROOT.parent / 'mfb_tc3_tc2',
    PROJECT_ROOT.parent / 'mfb_tabm_tc2',
    PROJECT_ROOT.parent / 'mfb_tc3_tc2',
]
PREVIOUS_EXPERIMENT_ROOTS = unique_paths([Path(x) for x in env_csv('PREVIOUS_EXPERIMENT_ROOTS', [])] + candidate_previous_roots)

if SMOKE:
    if 'DATASET_SEQUENCE' in os.environ:
        DATASET_SEQUENCE = env_csv('DATASET_SEQUENCE', TEAM_DATASETS[:1])
    else:
        DATASET_SEQUENCE = TEAM_DATASETS[:1]
    if 'FINAL_SEEDS' in os.environ:
        FINAL_SEEDS = env_int_list('FINAL_SEEDS', [0])
    else:
        FINAL_SEEDS = [0]
    SCREEN_SEED = FINAL_SEEDS[0]
    MAX_EPOCHS = min(MAX_EPOCHS, 5)
    DEBUG_MAX_TRAIN_ROWS = DEBUG_MAX_TRAIN_ROWS or 4096
    DEBUG_MAX_VAL_ROWS = DEBUG_MAX_VAL_ROWS or 2048
    DEBUG_MAX_TEST_ROWS = DEBUG_MAX_TEST_ROWS or 2048
    FEATURE_SCORE_BATCHES = min(FEATURE_SCORE_BATCHES, 2)
    FEATURE_SCORE_MAX_SAMPLES = min(FEATURE_SCORE_MAX_SAMPLES, 4096)
    ENABLE_ROUTER_SCREEN = False
else:
    DATASET_SEQUENCE = env_csv('DATASET_SEQUENCE', TEAM_DATASETS)
candidate_prepared_roots = [
    WORK_ROOT / 'prepared',
    WORK_ROOT / 'code' / 'tabred_official' / 'data',
    PROJECT_ROOT / 'prepared',
    PROJECT_ROOT / 'code' / 'tabred_official' / 'data',
    PROJECT_ROOT / 'person_c_strict_tabred_tc2' / 'prepared',
    PROJECT_ROOT / 'person_c_strict_tabred_tc2' / 'code' / 'tabred_official' / 'data',
    PROJECT_ROOT / 'person_c_strict_source_submission_bundle' / 'prepared',
    PROJECT_ROOT.parent / 'person_c_strict_tabred_tc2' / 'prepared',
    PROJECT_ROOT.parent / 'person_c_strict_tabred_tc2' / 'code' / 'tabred_official' / 'data',
    PROJECT_ROOT.parent / 'person_c_strict_source_submission_bundle' / 'prepared',
]
PREPARED_SEARCH_ROOTS = unique_paths([Path(x) for x in env_csv('EXISTING_PREPARED_ROOTS', [])] + candidate_prepared_roots)

TABM_REPO_CANDIDATES = unique_paths(
    [Path(x) for x in env_csv('TABM_REPO_ROOTS', [])]
    + [
        PROJECT_ROOT / 'tabm',
        PROJECT_ROOT.parent / 'tabm',
        WORK_ROOT / 'tabm',
        WORK_ROOT.parent / 'tabm',
        Path.home() / 'tabm',
    ]
)


def resolve_tabm_repo_root() -> Path:
    for candidate in TABM_REPO_CANDIDATES:
        root = candidate if candidate.name == 'tabm' else candidate / 'tabm'
        if root.joinpath('paper', 'pixi.toml').exists():
            return root.resolve()
    raise FileNotFoundError(f'TabM repo with paper/pixi.toml not found in {[str(x) for x in TABM_REPO_CANDIDATES]}')


TABM_REPO_ROOT = resolve_tabm_repo_root()
TABM_PAPER_DIR = TABM_REPO_ROOT / 'paper'
PAPER_TEMPLATE_ROOT = TABM_PAPER_DIR / 'exp' / 'tabm-piecewiselinear' / 'tabred'
PINNED_TABM_COMMIT = subprocess.check_output(['git', '-C', str(TABM_REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()

DATASET_META = {
    'homesite': {
        'prepared_dir': 'homesite-insurance',
        'aliases': ['homesite', 'homesite-insurance'],
        'canonical_rows': 260753,
    },
    'sberbank': {
        'prepared_dir': 'sberbank-housing',
        'aliases': ['sberbank', 'sberbank-housing'],
        'canonical_rows': 28321,
    },
    'ecom-offers': {
        'prepared_dir': 'ecom-offers',
        'aliases': ['ecom-offers'],
        'canonical_rows': 160057,
    },
    'cooking-time': {
        'prepared_dir': 'cooking-time',
        'aliases': ['cooking-time'],
        'canonical_rows': 319986,
    },
    'delivery-eta': {
        'prepared_dir': 'delivery-eta',
        'aliases': ['delivery-eta'],
        'canonical_rows': 350516,
    },
}


def load_paper_template(dataset_key: str) -> dict[str, Any]:
    template_path = PAPER_TEMPLATE_ROOT / DATASET_META[dataset_key]['prepared_dir'] / '0-evaluation' / '0.toml'
    if not template_path.exists():
        raise FileNotFoundError(f'Paper template not found: {template_path}')
    with template_path.open('rb') as f:
        return tomllib.load(f)


def build_dataset_registry() -> dict[str, dict[str, Any]]:
    registry: dict[str, dict[str, Any]] = {}
    for dataset_key, meta in DATASET_META.items():
        template = load_paper_template(dataset_key)
        task_type = 'regression' if 'regression' in meta['prepared_dir'] or dataset_key in {'sberbank', 'cooking-time', 'delivery-eta'} else 'binclass'
        if dataset_key in {'homesite', 'ecom-offers'}:
            task_type = 'binclass'
        model_cfg = template['model']
        backbone_cfg = model_cfg['backbone']
        num_embeddings_cfg = model_cfg.get('num_embeddings')
        registry[dataset_key] = {
            'prepared_dir': meta['prepared_dir'],
            'aliases': meta['aliases'],
            'task_type': task_type,
            'score_name': 'rmse' if task_type == 'regression' else 'roc-auc',
            'canonical_rows': meta['canonical_rows'],
            'template_path': str(PAPER_TEMPLATE_ROOT / meta['prepared_dir'] / '0-evaluation' / '0.toml'),
            'template_config': template,
            'batch_size': int(template['batch_size']),
            'patience': int(template['patience']),
            'grad_clip': float(template['gradient_clipping_norm']),
            'lr': float(template['optimizer']['lr']),
            'weight_decay': float(template['optimizer']['weight_decay']),
            'n_blocks': int(backbone_cfg['n_blocks']),
            'd_block': int(backbone_cfg['d_block']),
            'dropout': float(backbone_cfg['dropout']),
            'd_embedding': 0 if num_embeddings_cfg is None else int(num_embeddings_cfg['d_embedding']),
            'n_bins': int(template['bins']['n_bins']),
            'num_policy': template['data'].get('num_policy'),
            'cat_policy': template['data'].get('cat_policy'),
            'share_training_batches': bool(model_cfg.get('share_training_batches', False)),
            'amp': bool(template.get('amp', False)),
        }
    return registry


DATASET_REGISTRY = build_dataset_registry()

def make_variant_spec(
    display_name: str,
    *,
    family: str = 'legacy',
    mask_mode: str,
    mask_granularity: str = 'feature_group',
    keep_rate: float = 1.0,
    training_only: bool = False,
    inverted_scaling: bool = False,
    use_soft_mask: bool = False,
    mask_strength: float = 1.0,
    anchor_fraction: float = 0.0,
    core_fraction: float = 0.0,
    warmup_epochs: int = 0,
    use_baseline_init: bool = False,
    lr_scale: float = 1.0,
    max_epochs_override: int | None = None,
    patience_override: int | None = None,
    budget_target: float = 1.0,
    core_floor: float = 0.30,
    lambda_budget: float = 0.01,
    lambda_gate_div: float = 0.005,
    lambda_ncl: float = 0.0,
    lambda_route: float = 0.05,
    lambda_fidelity: float = 0.10,
    fidelity_warmup_fraction: float = 0.40,
    residual_rho: float = 0.50,
    use_router: bool = False,
    router_top_k: int = 2,
    router_tau: float = 1.0,
    router_hidden_dim: int = 64,
    share_training_batches_override: bool | None = None,
    aux_shared_batch_size: int = 512,
    aux_shared_interval: int = 1,
    amp_override: bool | None = None,
    dataset_overrides: dict[str, dict[str, Any]] | None = None,
) -> dict[str, Any]:
    return {
        'display_name': display_name,
        'family': family,
        'mask_mode': mask_mode,
        'mask_granularity': mask_granularity,
        'keep_rate': float(keep_rate),
        'training_only': bool(training_only),
        'inverted_scaling': bool(inverted_scaling),
        'use_soft_mask': bool(use_soft_mask),
        'mask_strength': float(mask_strength),
        'anchor_fraction': float(anchor_fraction),
        'core_fraction': float(core_fraction),
        'warmup_epochs': int(warmup_epochs),
        'use_baseline_init': bool(use_baseline_init),
        'lr_scale': float(lr_scale),
        'max_epochs_override': max_epochs_override,
        'patience_override': patience_override,
        'budget_target': float(budget_target),
        'core_floor': float(core_floor),
        'lambda_budget': float(lambda_budget),
        'lambda_gate_div': float(lambda_gate_div),
        'lambda_ncl': float(lambda_ncl),
        'lambda_route': float(lambda_route),
        'lambda_fidelity': float(lambda_fidelity),
        'fidelity_warmup_fraction': float(fidelity_warmup_fraction),
        'residual_rho': float(residual_rho),
        'use_router': bool(use_router),
        'router_top_k': int(router_top_k),
        'router_tau': float(router_tau),
        'router_hidden_dim': int(router_hidden_dim),
        'share_training_batches_override': share_training_batches_override,
        'aux_shared_batch_size': int(aux_shared_batch_size),
        'aux_shared_interval': int(aux_shared_interval),
        'amp_override': amp_override,
        'dataset_overrides': {} if dataset_overrides is None else deepcopy(dataset_overrides),
    }


VARIANT_REGISTRY = {
    'tabm_baseline': make_variant_spec(
        'TabM baseline',
        family='legacy',
        mask_mode='none',
    ),
    'mfb_feature_group_p05': make_variant_spec(
        'Legacy hard MFB p=0.5',
        family='legacy',
        mask_mode='member_fixed',
        keep_rate=0.5,
        inverted_scaling=True,
    ),
    'mfb_feature_group_p07': make_variant_spec(
        'Legacy hard MFB p=0.7',
        family='legacy',
        mask_mode='member_fixed',
        keep_rate=0.7,
        inverted_scaling=True,
    ),
    'mfb_feature_group_p09': make_variant_spec(
        'Legacy hard MFB p=0.9',
        family='legacy',
        mask_mode='member_fixed',
        keep_rate=0.9,
        inverted_scaling=True,
    ),
    'capmfb_safe_anchor50_p95_a025_core30': make_variant_spec(
        'CAP-MFB safe anchor=50% p=0.95 alpha=0.25 core=30%',
        family='legacy',
        mask_mode='member_fixed',
        keep_rate=0.95,
        use_soft_mask=True,
        mask_strength=0.25,
        anchor_fraction=0.50,
        core_fraction=0.30,
        warmup_epochs=8,
    ),
    'capmfb_safe_anchor25_p90_a05_core30': make_variant_spec(
        'CAP-MFB safe anchor=25% p=0.90 alpha=0.50 core=30%',
        family='legacy',
        mask_mode='member_fixed',
        keep_rate=0.90,
        use_soft_mask=True,
        mask_strength=0.50,
        anchor_fraction=0.25,
        core_fraction=0.30,
        warmup_epochs=6,
    ),
    'capmfb_eta_anchor25_p80_a05_core20': make_variant_spec(
        'CAP-MFB ETA anchor=25% p=0.80 alpha=0.50 core=20%',
        family='legacy',
        mask_mode='member_fixed',
        keep_rate=0.80,
        use_soft_mask=True,
        mask_strength=0.50,
        anchor_fraction=0.25,
        core_fraction=0.20,
        warmup_epochs=5,
    ),
    'ramp_mfb_no_ncl': make_variant_spec(
        'RAMP-MFB++ without NCL',
        family='ramp',
        mask_mode='none',
        budget_target=0.90,
        core_floor=0.30,
        lambda_budget=0.01,
        lambda_gate_div=0.005,
        lambda_route=0.05,
        lambda_fidelity=0.10,
        fidelity_warmup_fraction=0.40,
        residual_rho=0.50,
        use_router=False,
        dataset_overrides={
            'homesite': {'budget_target': 1.0, 'core_floor': 0.35, 'lambda_fidelity': 0.20, 'residual_rho': 0.35},
            'cooking-time': {'budget_target': 0.98, 'core_floor': 0.35, 'lambda_fidelity': 0.20, 'residual_rho': 0.40},
            'delivery-eta': {'budget_target': 0.90, 'core_floor': 0.30, 'lambda_fidelity': 0.12, 'residual_rho': 0.50},
            'ecom-offers': {'budget_target': 0.55, 'core_floor': 0.10, 'lambda_fidelity': 0.05, 'residual_rho': 0.60},
            'sberbank': {'budget_target': 0.90, 'core_floor': 0.20, 'lambda_fidelity': 0.05, 'residual_rho': 0.55},
        },
    ),
    'ramp_mfb_ncl_low': make_variant_spec(
        'RAMP-MFB++ with light NCL',
        family='ramp',
        mask_mode='none',
        budget_target=0.90,
        core_floor=0.30,
        lambda_budget=0.01,
        lambda_gate_div=0.005,
        lambda_route=0.05,
        lambda_fidelity=0.10,
        fidelity_warmup_fraction=0.40,
        residual_rho=0.50,
        use_router=False,
        dataset_overrides={
            'homesite': {'budget_target': 1.0, 'core_floor': 0.35, 'lambda_ncl': 0.001, 'lambda_fidelity': 0.20, 'residual_rho': 0.30},
            'cooking-time': {'budget_target': 0.98, 'core_floor': 0.35, 'lambda_ncl': 0.20, 'lambda_fidelity': 0.20, 'residual_rho': 0.40},
            'delivery-eta': {'budget_target': 0.90, 'core_floor': 0.30, 'lambda_ncl': 0.20, 'lambda_fidelity': 0.12, 'residual_rho': 0.50},
            'ecom-offers': {'budget_target': 0.55, 'core_floor': 0.10, 'lambda_ncl': 0.005, 'lambda_fidelity': 0.05, 'residual_rho': 0.60},
            'sberbank': {'budget_target': 0.90, 'core_floor': 0.20, 'lambda_ncl': 0.20, 'lambda_fidelity': 0.05, 'residual_rho': 0.55},
        },
    ),
    'ramp_mfb_ncl_mid': make_variant_spec(
        'RAMP-MFB++ with medium NCL',
        family='ramp',
        mask_mode='none',
        budget_target=0.90,
        core_floor=0.30,
        lambda_budget=0.01,
        lambda_gate_div=0.005,
        lambda_route=0.05,
        lambda_fidelity=0.10,
        fidelity_warmup_fraction=0.40,
        residual_rho=0.50,
        use_router=False,
        dataset_overrides={
            'homesite': {'budget_target': 1.0, 'core_floor': 0.35, 'lambda_ncl': 0.005, 'lambda_fidelity': 0.20, 'residual_rho': 0.30},
            'cooking-time': {'budget_target': 0.98, 'core_floor': 0.35, 'lambda_ncl': 0.30, 'lambda_fidelity': 0.20, 'residual_rho': 0.40},
            'delivery-eta': {'budget_target': 0.90, 'core_floor': 0.30, 'lambda_ncl': 0.30, 'lambda_fidelity': 0.12, 'residual_rho': 0.50},
            'ecom-offers': {'budget_target': 0.55, 'core_floor': 0.10, 'lambda_ncl': 0.010, 'lambda_fidelity': 0.05, 'residual_rho': 0.60},
            'sberbank': {'budget_target': 0.90, 'core_floor': 0.20, 'lambda_ncl': 0.30, 'lambda_fidelity': 0.05, 'residual_rho': 0.55},
        },
    ),
    'ramp_mfb_ncl_router': make_variant_spec(
        'RAMP-MFB++ with NCL and router',
        family='ramp',
        mask_mode='none',
        budget_target=0.90,
        core_floor=0.30,
        lambda_budget=0.01,
        lambda_gate_div=0.005,
        lambda_route=0.05,
        lambda_fidelity=0.10,
        fidelity_warmup_fraction=0.40,
        residual_rho=0.50,
        use_router=True,
        router_top_k=4,
        router_tau=1.0,
        dataset_overrides={
            'homesite': {'use_router': False, 'budget_target': 1.0, 'core_floor': 0.35, 'lambda_ncl': 0.001, 'lambda_fidelity': 0.20, 'residual_rho': 0.30},
            'cooking-time': {'use_router': True, 'budget_target': 0.98, 'core_floor': 0.35, 'lambda_ncl': 0.20, 'lambda_fidelity': 0.20, 'residual_rho': 0.45, 'router_top_k': 4},
            'delivery-eta': {'use_router': True, 'budget_target': 0.90, 'core_floor': 0.30, 'lambda_ncl': 0.20, 'lambda_fidelity': 0.12, 'residual_rho': 0.55, 'router_top_k': 4},
            'ecom-offers': {'use_router': False, 'budget_target': 0.55, 'core_floor': 0.10, 'lambda_ncl': 0.005, 'lambda_fidelity': 0.05, 'residual_rho': 0.60},
            'sberbank': {'use_router': True, 'budget_target': 0.90, 'core_floor': 0.20, 'lambda_ncl': 0.20, 'lambda_fidelity': 0.05, 'residual_rho': 0.60, 'router_top_k': 4},
        },
    ),
}

HARD_BEST_VARIANT_MAP = {
    'homesite': 'mfb_feature_group_p09',
    'sberbank': 'mfb_feature_group_p09',
    'ecom-offers': 'mfb_feature_group_p05',
    'cooking-time': 'mfb_feature_group_p07',
    'delivery-eta': 'mfb_feature_group_p07',
}

CAP_SELECTED_VARIANT_MAP = {
    'homesite': 'capmfb_safe_anchor25_p90_a05_core30',
    'sberbank': 'mfb_feature_group_p09',
    'ecom-offers': 'mfb_feature_group_p05',
    'cooking-time': 'capmfb_safe_anchor25_p90_a05_core30',
    'delivery-eta': 'capmfb_eta_anchor25_p80_a05_core20',
}

LEGACY_LIBRARY_MAP = {
    dataset_key: list(dict.fromkeys(['tabm_baseline', HARD_BEST_VARIANT_MAP[dataset_key], CAP_SELECTED_VARIANT_MAP[dataset_key]]))
    for dataset_key in TEAM_DATASETS
}

CURRENT_SCREEN_VARIANT_MAP = {
    'homesite': ['ramp_mfb_no_ncl', 'ramp_mfb_ncl_low', 'ramp_mfb_ncl_mid'],
    'sberbank': ['ramp_mfb_no_ncl', 'ramp_mfb_ncl_low', 'ramp_mfb_ncl_mid', 'ramp_mfb_ncl_router'],
    'ecom-offers': ['ramp_mfb_no_ncl', 'ramp_mfb_ncl_low', 'ramp_mfb_ncl_mid'],
    'cooking-time': ['ramp_mfb_no_ncl', 'ramp_mfb_ncl_low', 'ramp_mfb_ncl_mid', 'ramp_mfb_ncl_router'],
    'delivery-eta': ['ramp_mfb_no_ncl', 'ramp_mfb_ncl_low', 'ramp_mfb_ncl_mid', 'ramp_mfb_ncl_router'],
}
if not ENABLE_ROUTER_SCREEN:
    CURRENT_SCREEN_VARIANT_MAP = {
        dataset_key: [name for name in values if name != 'ramp_mfb_ncl_router']
        for dataset_key, values in CURRENT_SCREEN_VARIANT_MAP.items()
    }


def resolve_variant_config(variant_name: str, dataset_key: str | None = None) -> dict[str, Any]:
    cfg = deepcopy(VARIANT_REGISTRY[variant_name])
    if dataset_key is not None:
        cfg.update(deepcopy(cfg.get('dataset_overrides', {}).get(dataset_key, {})))
    return cfg

missing_datasets = [x for x in DATASET_SEQUENCE if x not in DATASET_REGISTRY]
all_referenced_variants = {'tabm_baseline'}
for values in LEGACY_LIBRARY_MAP.values():
    all_referenced_variants.update(values)
for values in CURRENT_SCREEN_VARIANT_MAP.values():
    all_referenced_variants.update(values)
missing_variants = [x for x in sorted(all_referenced_variants) if x not in VARIANT_REGISTRY]
if missing_datasets:
    raise KeyError(f'Unsupported datasets: {missing_datasets}')
if missing_variants:
    raise KeyError(f'Unsupported variants: {missing_variants}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
run_header = {
    'project_root': str(PROJECT_ROOT),
    'work_root': str(WORK_ROOT),
    'artifact_root': str(ARTIFACT_ROOT),
    'team_experiment_root': str(TEAM_EXPERIMENT_ROOT),
    'tabm_repo_root': str(TABM_REPO_ROOT),
    'pinned_tabm_commit': PINNED_TABM_COMMIT,
    'prepared_search_roots': [str(x) for x in PREPARED_SEARCH_ROOTS if x.exists()],
    'datasets': DATASET_SEQUENCE,
    'screen_seed': SCREEN_SEED,
    'final_seeds': FINAL_SEEDS,
    'split_name': SPLIT_NAME,
    'max_epochs': MAX_EPOCHS,
    'feature_score_batches': FEATURE_SCORE_BATCHES,
    'feature_score_max_samples': FEATURE_SCORE_MAX_SAMPLES,
    'device': str(device),
    'torch': torch.__version__,
    'tabm': getattr(tabm, '__version__', 'unknown'),
    'legacy_library_map': LEGACY_LIBRARY_MAP,
    'current_screen_variant_map': CURRENT_SCREEN_VARIANT_MAP,
    'blend_regularization_grid': BLEND_REGULARIZATION_GRID,
    'classification_blend_methods': CLASSIFICATION_BLEND_METHODS,
    'router_enabled_default': ROUTER_ENABLED_DEFAULT,
}
print(json.dumps(run_header, indent=2))
write_json(ARTIFACT_ROOT / 'run_config.json', run_header)
def normalize_name(text: str) -> str:
    return ''.join(ch for ch in text.lower() if ch.isalnum())


def locate_prepared_dataset(dataset_key: str) -> Path:
    spec = DATASET_REGISTRY[dataset_key]
    valid_names = {spec['prepared_dir'], *spec['aliases']}
    normalized = {normalize_name(x) for x in valid_names}
    for root in PREPARED_SEARCH_ROOTS:
        if not root.exists():
            continue
        direct = root / spec['prepared_dir']
        if direct.exists() and direct.joinpath('info.json').exists():
            return direct
        for child in root.iterdir():
            if child.is_dir() and normalize_name(child.name) in normalized and child.joinpath('info.json').exists():
                return child
    raise FileNotFoundError(f'Prepared dataset for {dataset_key} not found in {[str(x) for x in PREPARED_SEARCH_ROOTS]}')


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def expected_manifest_paths(prepared_dir: Path) -> list[Path]:
    required = [prepared_dir / name for name in ['Y.npy', 'X_num.npy', 'X_bin.npy', 'X_cat.npy', 'info.json']]
    split_dir = resolve_split_dir(prepared_dir, SPLIT_NAME)
    required.extend(split_dir / f'{part}_idx.npy' for part in ['train', 'val', 'test'])
    return [path for path in required if path.exists() and path.suffix == '.npy']


def ensure_manifest(prepared_dir: Path, dataset_key: str) -> dict[str, Any]:
    manifest_path = prepared_dir / 'manifest.json'
    npy_paths = expected_manifest_paths(prepared_dir)
    sha_map = {str(path.relative_to(prepared_dir)).replace('\\', '/'): file_sha256(path) for path in npy_paths}
    row_counts = {
        part: int(np.load(resolve_split_dir(prepared_dir, SPLIT_NAME) / f'{part}_idx.npy').shape[0])
        for part in ['train', 'val', 'test']
    }
    total_rows = int(sum(row_counts.values()))
    canonical_rows = int(DATASET_REGISTRY[dataset_key]['canonical_rows'])
    if total_rows != canonical_rows:
        raise ValueError(f'Canonical row-count mismatch for {dataset_key}: expected {canonical_rows}, found {total_rows}')
    manifest_payload = {
        'dataset': dataset_key,
        'source': f'tabred_preprocessed/{prepared_dir.name} (split-{SPLIT_NAME}/*_idx.npy)',
        'sha256': sha_map,
        'row_counts': row_counts,
        'total_rows': total_rows,
        'canonical_rows': canonical_rows,
        'pinned_tabm_commit': PINNED_TABM_COMMIT,
    }
    if manifest_path.exists():
        existing = json.loads(manifest_path.read_text(encoding='utf-8'))
        if existing.get('sha256') != sha_map or existing.get('total_rows') != total_rows:
            raise ValueError(f'Manifest verification failed for {prepared_dir}')
        return existing
    write_json(manifest_path, manifest_payload)
    return manifest_payload


def resolve_split_dir(prepared_dir: Path, split_name: str) -> Path:
    split_dir_name = split_name if split_name.startswith('split-') else f'split-{split_name}'
    split_dir = prepared_dir / split_dir_name
    if not split_dir.exists():
        raise FileNotFoundError(f'Split directory not found: {split_dir}')
    return split_dir


def load_optional_array(path: Path) -> np.ndarray | None:
    if not path.exists():
        return None
    return np.load(path, allow_pickle=True)


def subset_array(array: np.ndarray | None, indices: np.ndarray, dtype: np.dtype) -> np.ndarray:
    if array is None:
        return np.zeros((len(indices), 0), dtype=dtype)
    value = array[indices]
    if value.ndim == 1:
        return value.astype(dtype, copy=False)
    return value.astype(dtype, copy=False)


def transform_num(x_num: dict[str, np.ndarray], policy: str | None, seed: int) -> dict[str, np.ndarray]:
    if x_num['train'].shape[1] == 0:
        return {k: v.astype(np.float32, copy=False) for k, v in x_num.items()}
    if policy is not None:
        if policy == 'standard':
            normalizer = skp.StandardScaler()
            x_num_train = x_num['train']
        elif policy == 'noisy-quantile':
            normalizer = skp.QuantileTransformer(
                n_quantiles=max(min(x_num['train'].shape[0] // 30, 1000), 10),
                output_distribution='normal',
                subsample=1_000_000_000,
                random_state=seed,
            )
            noise = np.random.RandomState(seed).normal(0.0, 1e-5, x_num['train'].shape).astype(np.float32)
            x_num_train = x_num['train'] + noise
        else:
            raise ValueError(f'Unknown numerical policy: {policy}')
        normalizer.fit(x_num_train)
        x_num = {k: normalizer.transform(v) for k, v in x_num.items()}
    x_num = {k: np.nan_to_num(v).astype(np.float32) for k, v in x_num.items()}
    mask = np.array([len(np.unique(column)) > 1 for column in x_num['train'].T], dtype=bool)
    if mask.ndim == 1 and mask.size:
        x_num = {k: v[:, mask] for k, v in x_num.items()}
    return x_num


def transform_cat(x_cat: dict[str, np.ndarray], policy: str | None) -> dict[str, np.ndarray]:
    if policy is None or x_cat['train'].shape[1] == 0:
        return {k: v.astype(np.int64, copy=False) for k, v in x_cat.items()}
    unknown_value = np.iinfo('int64').max - 3
    encoder = skp.OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=unknown_value, dtype=np.int64)
    encoder.fit(x_cat['train'])
    x_cat = {k: encoder.transform(v) for k, v in x_cat.items()}
    max_values = x_cat['train'].max(axis=0) if x_cat['train'].shape[1] else np.array([], dtype=np.int64)
    for part in ['val', 'test']:
        if x_cat[part].shape[1] == 0:
            continue
        for column_idx in range(x_cat[part].shape[1]):
            mask = x_cat[part][:, column_idx] == unknown_value
            if mask.any():
                x_cat[part][mask, column_idx] = max_values[column_idx] + 1
    return {k: v.astype(np.int64, copy=False) for k, v in x_cat.items()}


@dataclass(frozen=True)
class RegressionLabelStats:
    mean: float
    std: float


def standardize_labels(y: dict[str, np.ndarray]) -> tuple[dict[str, np.ndarray], RegressionLabelStats]:
    mean = float(y['train'].mean())
    std = float(y['train'].std())
    std = std if std > 0 else 1.0
    transformed = {k: ((v - mean) / std).astype(np.float32) for k, v in y.items()}
    return transformed, RegressionLabelStats(mean=mean, std=std)


def convert_binary_to_categorical(data: dict[str, dict[str, np.ndarray]]) -> dict[str, dict[str, np.ndarray]]:
    x_bin = data.get('x_bin')
    if x_bin is None or x_bin['train'].shape[1] == 0:
        data['x_cat'] = data.get('x_cat', {part: np.zeros((len(data['y'][part]), 0), dtype=np.int64) for part in data['y']})
        return data
    n_bin_features = x_bin['train'].shape[1]
    good_idx = [i for i in range(n_bin_features) if len(np.unique(x_bin['train'][:, i])) > 1]
    if len(good_idx) < n_bin_features:
        x_bin = {k: v[:, good_idx] for k, v in x_bin.items()}
    x_cat_existing = data.get('x_cat')
    if x_cat_existing is None:
        x_cat_existing = {part: np.zeros((len(data['y'][part]), 0), dtype=np.int64) for part in data['y']}
    data['x_cat'] = {
        part: np.column_stack([x_cat_existing[part], x_bin[part].astype(np.int64)])
        for part in data['y']
    }
    data.pop('x_bin', None)
    return data


def apply_debug_limits(data: dict[str, dict[str, np.ndarray]]) -> dict[str, dict[str, np.ndarray]]:
    limits = {'train': DEBUG_MAX_TRAIN_ROWS, 'val': DEBUG_MAX_VAL_ROWS, 'test': DEBUG_MAX_TEST_ROWS}
    for part, limit in limits.items():
        if limit and limit > 0 and len(data['y'][part]) > limit:
            for key in list(data.keys()):
                data[key][part] = data[key][part][:limit]
    return data


BUNDLE_CACHE: dict[tuple[str, str, int, int, int, int], dict[str, Any]] = {}


def prepare_dataset_bundle(dataset_key: str, split_name: str, seed: int) -> dict[str, Any]:
    cache_key = (dataset_key, split_name, seed, DEBUG_MAX_TRAIN_ROWS, DEBUG_MAX_VAL_ROWS, DEBUG_MAX_TEST_ROWS)
    if cache_key in BUNDLE_CACHE:
        return BUNDLE_CACHE[cache_key]

    spec = DATASET_REGISTRY[dataset_key]
    prepared_dir = locate_prepared_dataset(dataset_key)
    manifest = ensure_manifest(prepared_dir, dataset_key)
    split_dir = resolve_split_dir(prepared_dir, split_name)
    info = json.loads(prepared_dir.joinpath('info.json').read_text())
    idx = {part: np.load(split_dir / f'{part}_idx.npy') for part in ['train', 'val', 'test']}

    x_num_full = load_optional_array(prepared_dir / 'X_num.npy')
    x_bin_full = load_optional_array(prepared_dir / 'X_bin.npy')
    x_cat_full = load_optional_array(prepared_dir / 'X_cat.npy')
    y_full = np.load(prepared_dir / 'Y.npy')

    data = {
        'x_num': {part: subset_array(x_num_full, idx[part], np.float32) for part in idx},
        'x_bin': {part: subset_array(x_bin_full, idx[part], np.float32) for part in idx},
        'x_cat': {part: subset_array(x_cat_full, idx[part], np.int64) for part in idx},
        'y': {part: subset_array(y_full, idx[part], np.float32 if info['task_type'] == 'regression' else np.int64) for part in idx},
    }
    data = convert_binary_to_categorical(data)
    data = apply_debug_limits(data)

    if data['x_num']['train'].shape[1] > 0:
        data['x_num'] = transform_num(data['x_num'], spec['num_policy'], seed)
    else:
        data['x_num'] = {k: v.astype(np.float32, copy=False) for k, v in data['x_num'].items()}

    if data['x_cat']['train'].shape[1] > 0:
        data['x_cat'] = transform_cat(data['x_cat'], spec['cat_policy'])
    else:
        data['x_cat'] = {k: v.astype(np.int64, copy=False) for k, v in data['x_cat'].items()}

    regression_stats = None
    if info['task_type'] == 'regression':
        data['y'], regression_stats = standardize_labels({k: v.astype(np.float32, copy=False) for k, v in data['y'].items()})
    else:
        data['y'] = {k: v.astype(np.int64, copy=False) for k, v in data['y'].items()}

    cat_cardinalities = []
    if data['x_cat']['train'].shape[1] > 0:
        for col in range(data['x_cat']['train'].shape[1]):
            cat_cardinalities.append(int(len(np.unique(data['x_cat']['train'][:, col]))))

    bin_edges = None
    if data['x_num']['train'].shape[1] > 0 and spec['d_embedding'] > 0:
        bin_edges = rtdl_num_embeddings.compute_bins(torch.as_tensor(data['x_num']['train'], dtype=torch.float32), n_bins=spec['n_bins'])

    tensors = {
        part: {
            'x_num': torch.from_numpy(data['x_num'][part].astype(np.float32, copy=False)),
            'x_cat': torch.from_numpy(data['x_cat'][part].astype(np.int64, copy=False)),
            'y': torch.from_numpy(data['y'][part]),
        }
        for part in ['train', 'val', 'test']
    }

    bundle = {
        'dataset_key': dataset_key,
        'prepared_dir': str(prepared_dir),
        'manifest': manifest,
        'split_name': split_name,
        'task_type': info['task_type'],
        'score_name': spec['score_name'],
        'x_num': data['x_num'],
        'x_cat': data['x_cat'],
        'y': data['y'],
        'tensors': tensors,
        'n_num_features': int(data['x_num']['train'].shape[1]),
        'cat_cardinalities': cat_cardinalities,
        'regression_stats': regression_stats,
        'bin_edges': bin_edges,
        'config': spec,
        'sizes': {part: len(data['y'][part]) for part in ['train', 'val', 'test']},
    }
    BUNDLE_CACHE[cache_key] = bundle
    return bundle


@dataclass(frozen=True)
class FeatureSpan:
    name: str
    start: int
    end: int
    kind: str


def mean_pairwise_jaccard(mask: np.ndarray) -> float:
    if mask.shape[0] < 2:
        return float('nan')
    values = []
    mask = mask.astype(bool)
    for i in range(mask.shape[0]):
        for j in range(i + 1, mask.shape[0]):
            union = np.logical_or(mask[i], mask[j]).sum()
            if union == 0:
                values.append(1.0)
            else:
                values.append(float(np.logical_and(mask[i], mask[j]).sum() / union))
    return float(np.mean(values))


def mask_coverage_stats(mask: np.ndarray) -> dict[str, float]:
    features_per_member = mask.sum(axis=1)
    members_per_feature = mask.sum(axis=0)
    return {
        'keep_rate_actual': float(mask.mean()),
        'min_features_per_member': float(features_per_member.min()),
        'mean_features_per_member': float(features_per_member.mean()),
        'max_features_per_member': float(features_per_member.max()),
        'min_members_per_feature': float(members_per_feature.min()),
        'mean_members_per_feature': float(members_per_feature.mean()),
        'max_members_per_feature': float(members_per_feature.max()),
        'mean_pairwise_jaccard': mean_pairwise_jaccard(mask),
    }


def mean_pairwise_corr(values: np.ndarray) -> float:
    if values.ndim != 2 or values.shape[1] < 2:
        return float('nan')
    corrs = []
    for i in range(values.shape[1]):
        xi = values[:, i]
        std_i = float(np.std(xi))
        for j in range(i + 1, values.shape[1]):
            xj = values[:, j]
            std_j = float(np.std(xj))
            if std_i < 1e-12 or std_j < 1e-12:
                corr = 1.0 if np.allclose(xi, xj) else 0.0
            else:
                corr = float(np.corrcoef(xi, xj)[0, 1])
            corrs.append(corr)
    return float(np.mean(corrs)) if corrs else float('nan')


def mean_pairwise_disagreement(class_predictions: np.ndarray) -> float:
    if class_predictions.ndim != 2 or class_predictions.shape[1] < 2:
        return float('nan')
    values = []
    for i in range(class_predictions.shape[1]):
        for j in range(i + 1, class_predictions.shape[1]):
            values.append(float(np.mean(class_predictions[:, i] != class_predictions[:, j])))
    return float(np.mean(values)) if values else float('nan')


def mean_pairwise_binary_kl(member_probs: np.ndarray, eps: float = 1e-12) -> float:
    if member_probs.ndim != 3 or member_probs.shape[1] < 2:
        return float('nan')
    values = []
    for i in range(member_probs.shape[1]):
        p = np.clip(member_probs[:, i, :], eps, 1.0)
        for j in range(i + 1, member_probs.shape[1]):
            q = np.clip(member_probs[:, j, :], eps, 1.0)
            kl_pq = np.mean(np.sum(p * np.log(p / q), axis=1))
            kl_qp = np.mean(np.sum(q * np.log(q / p), axis=1))
            values.append(float(0.5 * (kl_pq + kl_qp)))
    return float(np.mean(values)) if values else float('nan')


class OneHotEncodingModule(nn.Module):
    def __init__(self, cardinalities: list[int]) -> None:
        super().__init__()
        self.cardinalities = list(cardinalities)

    def forward(self, x_cat: torch.Tensor) -> torch.Tensor:
        if x_cat.ndim != 2:
            raise ValueError(f'Expected x_cat to have shape (B, C), got {tuple(x_cat.shape)}')
        if not self.cardinalities:
            return torch.zeros((x_cat.shape[0], 0), dtype=torch.float32, device=x_cat.device)
        parts = [
            F.one_hot(x_cat[:, i], num_classes=c + 1)[..., :-1]
            for i, c in enumerate(self.cardinalities)
        ]
        return torch.cat(parts, dim=1).to(torch.float32)


class MFBInputEncoder(nn.Module):
    def __init__(self, n_num_features: int, cat_cardinalities: list[int], num_embeddings: nn.Module | None) -> None:
        super().__init__()
        self.n_num_features = n_num_features
        self.cat_cardinalities = list(cat_cardinalities)
        self.num_module = num_embeddings
        self.cat_module = OneHotEncodingModule(self.cat_cardinalities) if self.cat_cardinalities else None
        self.feature_spans: list[FeatureSpan] = []
        cursor = 0
        if self.num_module is None:
            d_num_feature = 1
        else:
            output_shape = self.num_module.get_output_shape()
            if output_shape[0] != self.n_num_features:
                raise ValueError('Numerical embedding module was created for a different number of features')
            d_num_feature = int(output_shape[1])
        for idx in range(self.n_num_features):
            span = FeatureSpan(name=f'num_{idx}', start=cursor, end=cursor + d_num_feature, kind='num')
            self.feature_spans.append(span)
            cursor = span.end
        for idx, cardinality in enumerate(self.cat_cardinalities):
            span = FeatureSpan(name=f'cat_{idx}', start=cursor, end=cursor + int(cardinality), kind='cat')
            self.feature_spans.append(span)
            cursor = span.end
        self.d_out = cursor

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        parts = []
        if x_num.shape[1] > 0:
            if self.num_module is None:
                parts.append(x_num)
            else:
                parts.append(self.num_module(x_num).flatten(1))
        if x_cat.shape[1] > 0:
            if self.cat_module is None:
                raise RuntimeError('Categorical module is missing despite categorical inputs being present')
            parts.append(self.cat_module(x_cat))
        if not parts:
            raise RuntimeError('At least one input feature group must be present')
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=1)


def build_feature_index(feature_spans: list[FeatureSpan]) -> torch.Tensor:
    index = []
    for feature_id, span in enumerate(feature_spans):
        index.extend([feature_id] * (span.end - span.start))
    return torch.tensor(index, dtype=torch.long)


def make_feature_group_mask(
    k: int,
    feature_spans: list[FeatureSpan],
    keep_rate: float,
    seed: int,
    anchor_fraction: float = 0.0,
    protected_feature_ids: np.ndarray | None = None,
    ensure_each_feature_seen: bool = True,
    ensure_each_member_nonempty: bool = True,
) -> tuple[np.ndarray, np.ndarray, dict[str, float]]:
    feature_index = build_feature_index(feature_spans).numpy()
    n_features = len(feature_spans)
    gen = np.random.RandomState(seed)
    feature_mask = (gen.rand(k, n_features) < keep_rate).astype(np.float32)
    protected_feature_ids = np.array([], dtype=np.int64) if protected_feature_ids is None else np.asarray(protected_feature_ids, dtype=np.int64)
    n_anchor = int(round(k * anchor_fraction))
    n_anchor = max(0, min(k, n_anchor))
    if protected_feature_ids.size > 0:
        feature_mask[:, protected_feature_ids] = 1.0
    if n_anchor > 0:
        feature_mask[:n_anchor, :] = 1.0
    if ensure_each_member_nonempty:
        for row in range(k):
            if feature_mask[row].sum() == 0:
                feature_mask[row, gen.randint(0, n_features)] = 1.0
    if ensure_each_feature_seen:
        for col in range(n_features):
            if feature_mask[:, col].sum() == 0:
                feature_mask[gen.randint(0, k), col] = 1.0
    dim_mask = feature_mask[:, feature_index]
    stats = mask_coverage_stats(feature_mask)
    stats.update({
        'anchor_fraction_configured': float(anchor_fraction),
        'n_anchor_members': float(n_anchor),
        'core_fraction_configured': float(len(protected_feature_ids) / max(1, n_features)),
        'n_core_features': float(len(protected_feature_ids)),
    })
    return feature_mask.astype(np.float32), dim_mask.astype(np.float32), stats
def make_dimension_mask(k: int, d_encoded: int, keep_rate: float, seed: int, ensure_each_member_nonempty: bool = True, ensure_each_feature_seen: bool = True) -> tuple[np.ndarray | None, np.ndarray, dict[str, float]]:
    gen = np.random.RandomState(seed)
    dim_mask = (gen.rand(k, d_encoded) < keep_rate).astype(np.float32)
    if ensure_each_member_nonempty:
        for row in range(k):
            if dim_mask[row].sum() == 0:
                dim_mask[row, gen.randint(0, d_encoded)] = 1.0
    if ensure_each_feature_seen:
        for col in range(d_encoded):
            if dim_mask[:, col].sum() == 0:
                dim_mask[gen.randint(0, k), col] = 1.0
    stats = mask_coverage_stats(dim_mask)
    return None, dim_mask.astype(np.float32), stats


class MFBTabM(nn.Module):
    def __init__(
        self,
        *,
        n_num_features: int,
        cat_cardinalities: list[int],
        d_out: int,
        num_embeddings: nn.Module | None,
        n_blocks: int,
        d_block: int,
        dropout: float,
        k: int,
        mask_mode: str,
        mask_granularity: str,
        keep_rate: float,
        inverted_scaling: bool,
        mask_seed: int,
        mask_strength: float = 1.0,
        anchor_fraction: float = 0.0,
        protected_feature_ids: np.ndarray | None = None,
        warmup_epochs: int = 0,
        use_soft_mask: bool = False,
        share_training_batches: bool = False,
    ) -> None:
        super().__init__()
        self.k = k
        self.share_training_batches = bool(share_training_batches)
        self.mask_mode = mask_mode
        self.mask_granularity = mask_granularity
        self.keep_rate = keep_rate
        self.inverted_scaling = inverted_scaling
        self.mask_strength = float(mask_strength)
        self.anchor_fraction = float(anchor_fraction)
        self.warmup_epochs = int(warmup_epochs)
        self.use_soft_mask = bool(use_soft_mask)
        self.current_epoch = 0
        self.input_encoder = MFBInputEncoder(n_num_features=n_num_features, cat_cardinalities=cat_cardinalities, num_embeddings=num_embeddings)
        self.n_features = len(self.input_encoder.feature_spans)
        protected_feature_ids = np.array([], dtype=np.int64) if protected_feature_ids is None else np.asarray(protected_feature_ids, dtype=np.int64)
        self.register_buffer('protected_feature_ids', torch.from_numpy(protected_feature_ids.astype(np.int64, copy=False)), persistent=True)
        self.register_buffer('feature_index', build_feature_index(self.input_encoder.feature_spans), persistent=False)
        self.ensemble_view = tabm.EnsembleView(k=k)

        if mask_mode == 'member_fixed':
            if mask_granularity == 'feature_group':
                feature_mask, dim_mask, mask_stats = make_feature_group_mask(
                    k=k,
                    feature_spans=self.input_encoder.feature_spans,
                    keep_rate=keep_rate,
                    seed=mask_seed,
                    anchor_fraction=anchor_fraction,
                    protected_feature_ids=protected_feature_ids,
                )
            elif mask_granularity == 'dimension':
                feature_mask, dim_mask, mask_stats = make_dimension_mask(k=k, d_encoded=self.input_encoder.d_out, keep_rate=keep_rate, seed=mask_seed)
            else:
                raise ValueError(f'Unknown mask granularity: {mask_granularity}')
            fixed_feature_mask = np.empty((0, 0), dtype=np.float32) if feature_mask is None else feature_mask
            self.register_buffer('fixed_feature_mask', torch.from_numpy(fixed_feature_mask), persistent=True)
            self.register_buffer('fixed_dim_mask', torch.from_numpy(dim_mask), persistent=True)
            self.mask_stats = mask_stats
        else:
            self.register_buffer('fixed_feature_mask', torch.empty((0, 0), dtype=torch.float32), persistent=True)
            self.register_buffer('fixed_dim_mask', torch.empty((0, 0), dtype=torch.float32), persistent=True)
            self.mask_stats = {
                'keep_rate_actual': 1.0,
                'min_features_per_member': float(self.n_features),
                'mean_features_per_member': float(self.n_features),
                'max_features_per_member': float(self.n_features),
                'min_members_per_feature': float(k),
                'mean_members_per_feature': float(k),
                'max_members_per_feature': float(k),
                'mean_pairwise_jaccard': 1.0,
                'anchor_fraction_configured': float(anchor_fraction),
                'n_anchor_members': float(round(k * anchor_fraction)),
                'core_fraction_configured': float(len(protected_feature_ids) / max(1, self.n_features)),
                'n_core_features': float(len(protected_feature_ids)),
            }

        d_flat = self.input_encoder.d_out
        start_scaling_init = 'normal' if (num_embeddings is not None or len(cat_cardinalities) > 0) else 'random-signs'
        start_scaling_init_chunks = [span.end - span.start for span in self.input_encoder.feature_spans]
        self.backbone = tabm.make_tabm_backbone(
            d_in=d_flat,
            n_blocks=n_blocks,
            d_block=d_block,
            dropout=dropout,
            k=k,
            arch_type='tabm',
            start_scaling_init=start_scaling_init,
            start_scaling_init_chunks=start_scaling_init_chunks,
        )
        self.output = tabm.LinearEnsemble(d_block, d_out, k=k)

    def set_epoch(self, epoch: int) -> None:
        self.current_epoch = int(epoch)

    def _current_mask_strength(self) -> float:
        if self.warmup_epochs <= 0:
            return self.mask_strength
        progress = min(1.0, max(0.0, self.current_epoch / float(self.warmup_epochs)))
        return self.mask_strength * progress

    def _sample_stochastic_mask(self, device_: torch.device) -> torch.Tensor:
        if self.mask_granularity == 'feature_group':
            feature_mask = (torch.rand((self.k, self.n_features), device=device_) < self.keep_rate).to(torch.float32)
            zero_rows = feature_mask.sum(dim=1) == 0
            if bool(zero_rows.any()):
                zero_indices = torch.where(zero_rows)[0]
                random_cols = torch.randint(0, self.n_features, size=(len(zero_indices),), device=device_)
                feature_mask[zero_rows] = 0.0
                feature_mask[zero_indices, random_cols] = 1.0
            zero_cols = feature_mask.sum(dim=0) == 0
            if bool(zero_cols.any()):
                zero_indices = torch.where(zero_cols)[0]
                random_rows = torch.randint(0, self.k, size=(len(zero_indices),), device=device_)
                feature_mask[random_rows, zero_indices] = 1.0
            return feature_mask[:, self.feature_index]
        if self.mask_granularity == 'dimension':
            dim_mask = (torch.rand((self.k, self.input_encoder.d_out), device=device_) < self.keep_rate).to(torch.float32)
            zero_rows = dim_mask.sum(dim=1) == 0
            if bool(zero_rows.any()):
                zero_indices = torch.where(zero_rows)[0]
                random_cols = torch.randint(0, self.input_encoder.d_out, size=(len(zero_indices),), device=device_)
                dim_mask[zero_rows] = 0.0
                dim_mask[zero_indices, random_cols] = 1.0
            return dim_mask
        raise ValueError(f'Unknown mask granularity: {self.mask_granularity}')

    def _apply_mask(self, x: torch.Tensor, extra_dim_mask: torch.Tensor | None = None) -> torch.Tensor:
        if self.mask_mode == 'none' and extra_dim_mask is None:
            return x
        if self.mask_mode == 'member_fixed':
            raw_mask = self.fixed_dim_mask.to(device=x.device, dtype=x.dtype)
        elif self.mask_mode == 'stochastic':
            if not self.training and extra_dim_mask is None:
                return x
            raw_mask = self._sample_stochastic_mask(x.device).to(dtype=x.dtype)
        elif self.mask_mode == 'none':
            raw_mask = torch.ones((self.k, self.input_encoder.d_out), device=x.device, dtype=x.dtype)
        else:
            raise ValueError(f'Unknown mask mode: {self.mask_mode}')
        if extra_dim_mask is not None:
            if extra_dim_mask.ndim == 1:
                extra_dim_mask = extra_dim_mask.unsqueeze(0).expand_as(raw_mask)
            raw_mask = raw_mask * extra_dim_mask.to(device=x.device, dtype=x.dtype)
        if self.use_soft_mask and self.mask_mode == 'member_fixed':
            alpha = self._current_mask_strength()
            effective_mask = (1.0 - alpha) + alpha * raw_mask
            return x * effective_mask.unsqueeze(0)
        x = x * raw_mask.unsqueeze(0)
        if self.inverted_scaling and self.keep_rate < 1.0:
            x = x / max(self.keep_rate, 1e-6)
        return x

    def current_mask_snapshot(self) -> dict[str, np.ndarray | None]:
        feature_mask = None if self.fixed_feature_mask.numel() == 0 else self.fixed_feature_mask.detach().cpu().numpy()
        dim_mask = None if self.fixed_dim_mask.numel() == 0 else self.fixed_dim_mask.detach().cpu().numpy()
        return {'feature_mask': feature_mask, 'dim_mask': dim_mask}

    def encode_inputs(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        return self.input_encoder(x_num, x_cat)

    def prepare_ensemble_inputs(
        self,
        encoded: torch.Tensor,
        *,
        force_shared_batches: bool | None = None,
    ) -> torch.Tensor:
        use_shared_batches = self.share_training_batches if force_shared_batches is None else force_shared_batches
        if encoded.ndim == 3:
            if encoded.shape[1] != self.k:
                raise ValueError(f'Expected ensemble dimension {self.k}, got {tuple(encoded.shape)}')
            return encoded
        if encoded.ndim != 2:
            raise ValueError(f'Expected encoded input with 2 or 3 dimensions, got {tuple(encoded.shape)}')
        if force_shared_batches is None:
            if use_shared_batches or not self.training:
                return self.ensemble_view(encoded)
        elif use_shared_batches:
            return self.ensemble_view(encoded)
        if len(encoded) % self.k != 0:
            raise ValueError(f'Cannot reshape batch of size {len(encoded)} into k={self.k} members')
        return encoded.reshape(len(encoded) // self.k, self.k, encoded.shape[-1])

    def forward_encoded(
        self,
        encoded: torch.Tensor,
        extra_dim_mask: torch.Tensor | None = None,
        *,
        force_shared_batches: bool | None = None,
    ) -> torch.Tensor:
        x = self.prepare_ensemble_inputs(encoded, force_shared_batches=force_shared_batches)
        x = self._apply_mask(x, extra_dim_mask=extra_dim_mask)
        x = self.backbone(x)
        return self.output(x)

    def forward(
        self,
        x_num: torch.Tensor,
        x_cat: torch.Tensor,
        extra_dim_mask: torch.Tensor | None = None,
        *,
        force_shared_batches: bool | None = None,
    ) -> torch.Tensor:
        encoded = self.encode_inputs(x_num, x_cat)
        return self.forward_encoded(
            encoded,
            extra_dim_mask=extra_dim_mask,
            force_shared_batches=force_shared_batches,
        )


class LearnedFeatureGate(nn.Module):
    def __init__(self, *, k: int, feature_spans: list[FeatureSpan], init_logit: float = 2.0, core_floor: float = 0.30) -> None:
        super().__init__()
        self.k = int(k)
        self.feature_spans = list(feature_spans)
        self.n_features = len(self.feature_spans)
        self.core_floor = float(core_floor)
        self.logits = nn.Parameter(torch.full((self.k, self.n_features), float(init_logit)))
        self.register_buffer('feature_index', build_feature_index(self.feature_spans), persistent=False)

    def gates(self) -> tuple[torch.Tensor, torch.Tensor]:
        feature_gates = torch.sigmoid(self.logits)
        effective_feature_gates = self.core_floor + (1.0 - self.core_floor) * feature_gates
        effective_dim_gates = effective_feature_gates[:, self.feature_index]
        return effective_feature_gates, effective_dim_gates

    def budget_loss(self, budget_target: float) -> torch.Tensor:
        effective_feature_gates, _ = self.gates()
        target = self.core_floor + (1.0 - self.core_floor) * float(budget_target)
        return (effective_feature_gates.mean() - target) ** 2

    def diversity_loss(self) -> torch.Tensor:
        effective_feature_gates, _ = self.gates()
        if effective_feature_gates.shape[0] < 2:
            return effective_feature_gates.new_zeros(())
        normalized = F.normalize(effective_feature_gates, dim=1)
        cosine = normalized @ normalized.transpose(0, 1)
        upper = cosine[torch.triu(torch.ones_like(cosine, dtype=torch.bool), diagonal=1)]
        return upper.mean() if upper.numel() else cosine.new_zeros(())

    def snapshot(self) -> dict[str, np.ndarray]:
        effective_feature_gates, effective_dim_gates = self.gates()
        return {
            'feature_gates': effective_feature_gates.detach().cpu().numpy(),
            'dim_gates': effective_dim_gates.detach().cpu().numpy(),
            'raw_logits': self.logits.detach().cpu().numpy(),
        }


class SparseRouter(nn.Module):
    def __init__(self, *, d_input: int, n_experts: int, hidden_dim: int = 64, tau: float = 1.0, top_k: int = 2) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_input, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_experts),
        )
        self.tau = float(tau)
        self.top_k = int(top_k)
        self.n_experts = int(n_experts)

    def forward(self, encoded: torch.Tensor) -> torch.Tensor:
        logits = self.net(encoded) / max(self.tau, 1e-6)
        if self.top_k < self.n_experts:
            topk_values, topk_indices = logits.topk(self.top_k, dim=-1)
            sparse_logits = torch.full_like(logits, float('-inf'))
            sparse_logits.scatter_(1, topk_indices, topk_values)
            logits = sparse_logits
        return torch.softmax(logits, dim=-1)


def route_balance_loss(router_weights: torch.Tensor) -> torch.Tensor:
    if router_weights.numel() == 0:
        return router_weights.new_zeros(())
    mean_load = router_weights.mean(dim=0)
    target = torch.full_like(mean_load, 1.0 / mean_load.numel())
    return F.mse_loss(mean_load, target)


def route_entropy(router_weights: torch.Tensor) -> torch.Tensor:
    if router_weights.numel() == 0:
        return router_weights.new_zeros(())
    probs = router_weights.clamp_min(1e-8)
    return -(probs * probs.log()).sum(dim=-1).mean()


def weighted_member_average(member_logits: torch.Tensor, router_weights: torch.Tensor | None) -> torch.Tensor:
    if router_weights is None:
        return member_logits.mean(dim=1)
    weights = router_weights.unsqueeze(-1)
    return (member_logits * weights).sum(dim=1)


class RampResidualTabM(nn.Module):
    def __init__(
        self,
        *,
        baseline_anchor: MFBTabM,
        n_num_features: int,
        cat_cardinalities: list[int],
        d_out: int,
        num_embeddings: nn.Module | None,
        n_blocks: int,
        d_block: int,
        dropout: float,
        k: int,
        core_floor: float,
        residual_rho: float,
        use_router: bool,
        router_top_k: int,
        router_tau: float,
        router_hidden_dim: int,
        share_training_batches: bool,
    ) -> None:
        super().__init__()
        self.k = int(k)
        self.residual_rho = float(residual_rho)
        self.use_router = bool(use_router)
        self.share_training_batches = bool(share_training_batches)
        self.baseline_anchor = baseline_anchor
        for parameter in self.baseline_anchor.parameters():
            parameter.requires_grad_(False)
        self.baseline_anchor.eval()
        self.input_encoder = MFBInputEncoder(n_num_features=n_num_features, cat_cardinalities=cat_cardinalities, num_embeddings=num_embeddings)
        self.gates = LearnedFeatureGate(k=k, feature_spans=self.input_encoder.feature_spans, init_logit=2.0, core_floor=core_floor)
        self.ensemble_view = tabm.EnsembleView(k=k)
        start_scaling_init = 'normal' if (num_embeddings is not None or len(cat_cardinalities) > 0) else 'random-signs'
        start_scaling_init_chunks = [span.end - span.start for span in self.input_encoder.feature_spans]
        self.backbone = tabm.make_tabm_backbone(
            d_in=self.input_encoder.d_out,
            n_blocks=n_blocks,
            d_block=d_block,
            dropout=dropout,
            k=k,
            arch_type='tabm',
            start_scaling_init=start_scaling_init,
            start_scaling_init_chunks=start_scaling_init_chunks,
        )
        self.output = tabm.LinearEnsemble(d_block, d_out, k=k)
        self.router = None if not self.use_router else SparseRouter(
            d_input=self.input_encoder.d_out,
            n_experts=k,
            hidden_dim=router_hidden_dim,
            tau=router_tau,
            top_k=router_top_k,
        )

    def encode_inputs(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        return self.input_encoder(x_num, x_cat)

    def prepare_ensemble_inputs(
        self,
        encoded: torch.Tensor,
        *,
        force_shared_batches: bool | None = None,
    ) -> torch.Tensor:
        use_shared_batches = self.share_training_batches if force_shared_batches is None else force_shared_batches
        if encoded.ndim == 3:
            if encoded.shape[1] != self.k:
                raise ValueError(f'Expected ensemble dimension {self.k}, got {tuple(encoded.shape)}')
            return encoded
        if encoded.ndim != 2:
            raise ValueError(f'Expected encoded input with 2 or 3 dimensions, got {tuple(encoded.shape)}')
        if force_shared_batches is None:
            if use_shared_batches or not self.training:
                return self.ensemble_view(encoded)
        elif use_shared_batches:
            return self.ensemble_view(encoded)
        if len(encoded) % self.k != 0:
            raise ValueError(f'Cannot reshape batch of size {len(encoded)} into k={self.k} members')
        return encoded.reshape(len(encoded) // self.k, self.k, encoded.shape[-1])

    def forward_components(
        self,
        x_num: torch.Tensor,
        x_cat: torch.Tensor,
        *,
        force_shared_batches: bool | None = None,
    ) -> dict[str, torch.Tensor | None]:
        with torch.inference_mode():
            baseline_member_logits = self.baseline_anchor(
                x_num,
                x_cat,
                force_shared_batches=force_shared_batches,
            ).detach()
        baseline_ensemble_logits = baseline_member_logits.mean(dim=1)

        encoded = self.encode_inputs(x_num, x_cat)
        _, dim_gates = self.gates.gates()
        x = self.prepare_ensemble_inputs(encoded, force_shared_batches=force_shared_batches)
        x = x * dim_gates.unsqueeze(0)
        specialist_member_logits = self.output(self.backbone(x))
        final_member_logits = baseline_member_logits + self.residual_rho * (
            specialist_member_logits - baseline_member_logits
        )
        router_weights = None
        if self.router is not None and (force_shared_batches if force_shared_batches is not None else self.share_training_batches or not self.training):
            router_input = encoded if encoded.ndim == 2 else encoded.mean(dim=1)
            router_weights = self.router(router_input)
        specialist_ensemble_logits = weighted_member_average(specialist_member_logits, router_weights)
        final_ensemble_logits = weighted_member_average(final_member_logits, router_weights)
        return {
            'baseline_member_logits': baseline_member_logits,
            'baseline_ensemble_logits': baseline_ensemble_logits,
            'specialist_member_logits': specialist_member_logits,
            'specialist_ensemble_logits': specialist_ensemble_logits,
            'final_member_logits': final_member_logits,
            'final_ensemble_logits': final_ensemble_logits,
            'router_weights': router_weights,
        }

    def forward(
        self,
        x_num: torch.Tensor,
        x_cat: torch.Tensor,
        *,
        force_shared_batches: bool | None = None,
    ) -> torch.Tensor:
        return self.forward_components(
            x_num,
            x_cat,
            force_shared_batches=force_shared_batches,
        )['final_ensemble_logits']
def set_seed(seed: int) -> None:
    delu.random.seed(seed)


def default_zero_weight_decay_condition(
    module_name: str,
    module: nn.Module,
    parameter_name: str,
    parameter: nn.Parameter,
) -> bool:
    from rtdl_num_embeddings import _Periodic

    del module_name, parameter
    return parameter_name.endswith('bias') or isinstance(
        module,
        nn.BatchNorm1d
        | nn.LayerNorm
        | nn.InstanceNorm1d
        | rtdl_revisiting_models.LinearEmbeddings
        | rtdl_num_embeddings.LinearEmbeddings
        | rtdl_num_embeddings.LinearReLUEmbeddings
        | _Periodic,
    )


def make_parameter_groups(
    module: nn.Module,
    zero_weight_decay_condition=default_zero_weight_decay_condition,
    custom_groups: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    if custom_groups is None:
        custom_groups = []
    custom_params = frozenset(
        itertools.chain.from_iterable(group['params'] for group in custom_groups)
    )
    assert len(custom_params) == sum(
        len(group['params']) for group in custom_groups
    ), 'Parameters in custom_groups must not intersect'
    zero_wd_params = frozenset(
        p
        for mn, m in module.named_modules()
        for pn, p in m.named_parameters()
        if p not in custom_params and zero_weight_decay_condition(mn, m, pn, p)
    )
    default_group = {
        'params': [
            p
            for p in module.parameters()
            if p not in custom_params and p not in zero_wd_params
        ]
    }
    return [
        default_group,
        {'params': list(zero_wd_params), 'weight_decay': 0.0},
        *custom_groups,
    ]


def get_n_parameters(module: nn.Module) -> int:
    return sum(param.numel() for param in module.parameters() if param.requires_grad)


def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(seconds)))
    return str(timedelta(seconds=seconds))


def prediction_type_for_task(task_type: str) -> str:
    return 'probs' if task_type == 'binclass' else 'labels'


def amp_dtype_string(amp_dtype: torch.dtype | None) -> str:
    if amp_dtype is torch.bfloat16:
        return 'bfloat16'
    if amp_dtype is torch.float16:
        return 'float16'
    return 'fp32'


def evaluation_dir_name(variant_name: str, inference_mode: str) -> str:
    suffix = {
        'mean': '-evaluation',
        'best-head': '-best-head-evaluation',
        'greedy-heads': '-greedy-heads-evaluation',
        'router': '-router-evaluation',
    }[inference_mode]
    return f'{variant_name}{suffix}'


def run_dir_for(dataset_key: str, variant_name: str, seed: int, inference_mode: str = 'mean') -> Path:
    return TEAM_EXPERIMENT_ROOT / dataset_key / evaluation_dir_name(variant_name, inference_mode) / str(seed)


def write_toml(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(tomli_w.dumps(sanitize_toml_value(payload)), encoding='utf-8')


def sanitize_toml_value(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        sanitized = {}
        for key, item in value.items():
            clean_item = sanitize_toml_value(item)
            if clean_item is not None:
                sanitized[key] = clean_item
        return sanitized
    if isinstance(value, (list, tuple)):
        return [clean_item for item in value if (clean_item := sanitize_toml_value(item)) is not None]
    if isinstance(value, np.generic):
        return value.item()
    return value


def base_run_config(bundle: dict[str, Any], variant_name: str, seed: int) -> dict[str, Any]:
    variant_cfg = resolve_variant_config(variant_name, bundle['dataset_key'])
    model_cfg = {
        'arch_type': 'tabm',
        'k': 32,
        'share_training_batches': bool(
            variant_cfg.get('share_training_batches_override')
            if variant_cfg.get('share_training_batches_override') is not None
            else bundle['config']['share_training_batches']
        ),
    }
    config = {
        'seed': int(seed),
        'data': {
            'path': bundle['prepared_dir'],
            'num_policy': bundle['config']['num_policy'],
            'cat_policy': bundle['config']['cat_policy'],
            'split': bundle['split_name'],
        },
        'bins': {'n_bins': int(bundle['config']['n_bins'])},
        'model': model_cfg,
        'optimizer': {
            'type': 'AdamW',
            'lr': float(bundle['config']['lr'] * variant_cfg.get('lr_scale', 1.0)),
            'weight_decay': float(bundle['config']['weight_decay']),
        },
        'batch_size': int(bundle['config']['batch_size']),
        'eval_batch_size': int(EVAL_BATCH_SIZE),
        'patience': int(variant_cfg.get('patience_override') or bundle['config']['patience']),
        'n_epochs': int(variant_cfg.get('max_epochs_override') or MAX_EPOCHS),
        'gradient_clipping_norm': float(bundle['config']['grad_clip']),
        'amp': bool(
            variant_cfg.get('amp_override')
            if variant_cfg.get('amp_override') is not None
            else bundle['config']['amp']
        ),
        'variant_name': variant_name,
        'variant_display_name': variant_cfg['display_name'],
        'git_commit': PINNED_TABM_COMMIT,
    }
    return config


def finish_run_dir(
    *,
    bundle: dict[str, Any],
    variant_name: str,
    seed: int,
    inference_mode: str,
    config_payload: dict[str, Any],
    report_payload: dict[str, Any],
) -> Path:
    output_dir = run_dir_for(bundle['dataset_key'], variant_name, seed, inference_mode)
    output_dir.mkdir(parents=True, exist_ok=True)
    write_toml(output_dir / '0.toml', config_payload)
    write_json(output_dir / 'report.json', report_payload)
    (output_dir / 'DONE').write_text('', encoding='utf-8')
    return output_dir


def make_tensor_dataloaders(bundle: dict[str, Any], batch_size: int, eval_batch_size: int) -> dict[str, DataLoader]:
    loaders = {}
    for part, bs, shuffle in [('train', batch_size, True), ('val', eval_batch_size, False), ('test', eval_batch_size, False)]:
        dataset = TensorDataset(
            torch.from_numpy(bundle['x_num'][part].astype(np.float32, copy=False)),
            torch.from_numpy(bundle['x_cat'][part].astype(np.int64, copy=False)),
            torch.from_numpy(bundle['y'][part]),
        )
        loaders[part] = DataLoader(
            dataset,
            batch_size=bs,
            shuffle=shuffle,
            num_workers=NUM_WORKERS,
            pin_memory=(device.type == 'cuda'),
            persistent_workers=(NUM_WORKERS > 0),
            drop_last=False,
        )
    return loaders


def locate_run_dir(dataset_key: str, variant_name: str, seed: int) -> Path | None:
    candidate = run_dir_for(dataset_key, variant_name, seed, 'mean')
    if candidate.joinpath('DONE').exists():
        return candidate
    return candidate if candidate.exists() else None


def checkpoint_path_for(dataset_key: str, variant_name: str, seed: int) -> Path | None:
    run_dir = locate_run_dir(dataset_key, variant_name, seed)
    if run_dir is None:
        return None
    checkpoint_path = run_dir / 'checkpoint_best.pt'
    return checkpoint_path if checkpoint_path.exists() else None


def baseline_checkpoint_path(bundle: dict[str, Any], seed: int) -> Path:
    checkpoint_path = checkpoint_path_for(bundle['dataset_key'], 'tabm_baseline', seed)
    if checkpoint_path is None:
        raise FileNotFoundError(f'Baseline checkpoint not found for {bundle["dataset_key"]} seed {seed}')
    return checkpoint_path


def feature_saliency_cache_path(bundle: dict[str, Any], seed: int) -> Path:
    return CACHE_ROOT / 'feature_saliency' / bundle['dataset_key'] / f'seed{seed}.npz'


def load_matching_state(model: nn.Module, state_dict: dict[str, torch.Tensor]) -> int:
    current_state = model.state_dict()
    filtered = {
        key: value
        for key, value in state_dict.items()
        if key in current_state and current_state[key].shape == value.shape and not key.startswith('fixed_')
    }
    model.load_state_dict(filtered, strict=False)
    return len(filtered)


def compute_feature_saliency(bundle: dict[str, Any], seed: int) -> np.ndarray:
    cache_path = feature_saliency_cache_path(bundle, seed)
    if cache_path.exists() and not FORCE_RERUN:
        return np.load(cache_path, allow_pickle=True)['feature_scores']
    checkpoint_path = baseline_checkpoint_path(bundle, seed)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Baseline checkpoint required before saliency computation: {checkpoint_path}')
    baseline_model = build_model(bundle, 'tabm_baseline', seed)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    load_matching_state(baseline_model, checkpoint['model_state_dict'])
    baseline_model.eval()
    loaders = make_tensor_dataloaders(bundle, batch_size=bundle['config']['batch_size'], eval_batch_size=EVAL_BATCH_SIZE)
    dim_scores = np.zeros(baseline_model.input_encoder.d_out, dtype=np.float64)
    seen_batches = 0
    seen_samples = 0
    for x_num, x_cat, y in loaders['train']:
        x_num = x_num.to(device, non_blocking=True)
        x_cat = x_cat.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        encoded = baseline_model.encode_inputs(x_num, x_cat).detach().requires_grad_(True)
        logits = baseline_model.forward_encoded(encoded)
        loss = memberwise_loss(logits, y, bundle['task_type'])
        grad = torch.autograd.grad(loss, encoded)[0]
        dim_scores += grad.detach().abs().mean(dim=0).cpu().numpy()
        seen_batches += 1
        seen_samples += len(y)
        if seen_batches >= FEATURE_SCORE_BATCHES or seen_samples >= FEATURE_SCORE_MAX_SAMPLES:
            break
    if seen_batches == 0:
        raise RuntimeError(f'No batches available for feature saliency on {bundle["dataset_key"]}')
    dim_scores /= float(seen_batches)
    feature_scores = np.zeros(baseline_model.n_features, dtype=np.float64)
    feature_names = []
    for feature_id, span in enumerate(baseline_model.input_encoder.feature_spans):
        feature_names.append(span.name)
        values = dim_scores[span.start : span.end]
        feature_scores[feature_id] = float(np.max(values)) if len(values) else 0.0
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        feature_scores=feature_scores.astype(np.float32),
        dim_scores=dim_scores.astype(np.float32),
        feature_names=np.asarray(feature_names, dtype=object),
    )
    del baseline_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return feature_scores.astype(np.float32)


def resolve_core_feature_ids(bundle: dict[str, Any], seed: int, core_fraction: float) -> np.ndarray | None:
    if core_fraction <= 0:
        return None
    scores = compute_feature_saliency(bundle, seed)
    n_core = max(1, int(round(len(scores) * core_fraction)))
    ranked = np.argsort(-scores)
    return ranked[:n_core].astype(np.int64)


def create_num_embeddings(bundle: dict[str, Any]) -> nn.Module | None:
    spec = bundle['config']
    if bundle['n_num_features'] > 0 and spec['d_embedding'] > 0:
        return rtdl_num_embeddings.PiecewiseLinearEmbeddings(
            bundle['bin_edges'],
            d_embedding=spec['d_embedding'],
            activation=False,
            version='B',
        )
    return None


def build_legacy_model(bundle: dict[str, Any], variant_name: str, seed: int) -> MFBTabM:
    spec = bundle['config']
    variant = resolve_variant_config(variant_name, bundle['dataset_key'])
    share_training_batches = (
        variant.get('share_training_batches_override')
        if variant.get('share_training_batches_override') is not None
        else spec['share_training_batches']
    )
    num_embeddings = create_num_embeddings(bundle)
    d_out = 2 if bundle['task_type'] == 'binclass' else 1
    mask_seed = stable_int_from_parts('mask', bundle['dataset_key'], variant_name, seed)
    protected_feature_ids = None
    if variant['mask_mode'] == 'member_fixed' and variant['mask_granularity'] == 'feature_group' and variant['core_fraction'] > 0:
        protected_feature_ids = resolve_core_feature_ids(bundle, seed, variant['core_fraction'])
    model = MFBTabM(
        n_num_features=bundle['n_num_features'],
        cat_cardinalities=bundle['cat_cardinalities'],
        d_out=d_out,
        num_embeddings=num_embeddings,
        n_blocks=spec['n_blocks'],
        d_block=spec['d_block'],
        dropout=spec['dropout'],
        k=32,
        mask_mode=variant['mask_mode'],
        mask_granularity=variant['mask_granularity'],
        keep_rate=variant['keep_rate'],
        inverted_scaling=variant['inverted_scaling'],
        mask_seed=mask_seed,
        mask_strength=variant['mask_strength'],
        anchor_fraction=variant['anchor_fraction'],
        protected_feature_ids=protected_feature_ids,
        warmup_epochs=variant['warmup_epochs'],
        use_soft_mask=variant['use_soft_mask'],
        share_training_batches=share_training_batches,
    )
    return model.to(device)


def build_ramp_model(bundle: dict[str, Any], variant_name: str, seed: int) -> RampResidualTabM:
    spec = bundle['config']
    variant = resolve_variant_config(variant_name, bundle['dataset_key'])
    share_training_batches = (
        variant.get('share_training_batches_override')
        if variant.get('share_training_batches_override') is not None
        else spec['share_training_batches']
    )
    checkpoint_path = baseline_checkpoint_path(bundle, seed)
    baseline_anchor = build_legacy_model(bundle, 'tabm_baseline', seed)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    load_matching_state(baseline_anchor, checkpoint['model_state_dict'])
    baseline_anchor.eval()
    num_embeddings = create_num_embeddings(bundle)
    d_out = 2 if bundle['task_type'] == 'binclass' else 1
    model = RampResidualTabM(
        baseline_anchor=baseline_anchor,
        n_num_features=bundle['n_num_features'],
        cat_cardinalities=bundle['cat_cardinalities'],
        d_out=d_out,
        num_embeddings=num_embeddings,
        n_blocks=spec['n_blocks'],
        d_block=spec['d_block'],
        dropout=spec['dropout'],
        k=32,
        core_floor=variant['core_floor'],
        residual_rho=variant['residual_rho'],
        use_router=variant['use_router'] if ROUTER_ENABLED_DEFAULT or variant['use_router'] else False,
        router_top_k=variant['router_top_k'],
        router_tau=variant['router_tau'],
        router_hidden_dim=variant['router_hidden_dim'],
        share_training_batches=share_training_batches,
    )
    return model.to(device)


def build_model(bundle: dict[str, Any], variant_name: str, seed: int) -> nn.Module:
    variant = resolve_variant_config(variant_name, bundle['dataset_key'])
    if variant['family'] == 'legacy':
        return build_legacy_model(bundle, variant_name, seed)
    if variant['family'] == 'ramp':
        return build_ramp_model(bundle, variant_name, seed)
    raise ValueError(f'Unknown variant family: {variant["family"]}')


def memberwise_loss(logits: torch.Tensor, target: torch.Tensor, task_type: str) -> torch.Tensor:
    if task_type == 'binclass':
        target_expanded = target.long().repeat_interleave(logits.shape[1])
        return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), target_expanded)
    target = target.float().view(-1, 1)
    target_expanded = target[:, None, :].expand_as(logits)
    return F.mse_loss(logits, target_expanded)


def ensemble_task_loss(logits: torch.Tensor, target: torch.Tensor, task_type: str) -> torch.Tensor:
    if task_type == 'binclass':
        return F.cross_entropy(logits, target.long())
    return F.mse_loss(logits.view(-1), target.float().view(-1))


def ncl_diversity(logits: torch.Tensor, task_type: str) -> torch.Tensor:
    if task_type == 'binclass':
        probs = torch.softmax(logits, dim=-1)
        mean_probs = probs.mean(dim=1, keepdim=True)
        eps = 1e-8
        kl = (probs * (torch.log(probs + eps) - torch.log(mean_probs + eps))).sum(dim=-1)
        return kl.mean()
    mean_logits = logits.mean(dim=1, keepdim=True)
    return ((logits - mean_logits) ** 2).mean()


def fidelity_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, task_type: str) -> torch.Tensor:
    if task_type == 'binclass':
        student_log_probs = F.log_softmax(student_logits, dim=-1)
        teacher_probs = torch.softmax(teacher_logits, dim=-1).detach()
        return F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
    return F.mse_loss(student_logits.view(-1), teacher_logits.detach().view(-1))


def safe_roc_auc(y_true: np.ndarray, positive_probs: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float('nan')
    return float(skm.roc_auc_score(y_true, positive_probs))


def state_dict_to_cpu(model: nn.Module) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def save_array_bundle(path: Path, payload: dict[str, np.ndarray]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **payload)


def metric_key_from_score_name(score_name: str) -> str:
    return score_name.replace('-', '_')


def signed_score(score_name: str, value: float) -> float:
    return float(value if score_name == 'roc-auc' else -value)


def relative_improvement(metric_name: str, baseline_value: float, candidate_value: float) -> float:
    if metric_name == 'roc-auc':
        return float((candidate_value - baseline_value) / max(abs(baseline_value), 1e-12) * 100.0)
    return float((baseline_value - candidate_value) / max(abs(baseline_value), 1e-12) * 100.0)


def raw_run_dir(dataset_key: str, variant_name: str, seed: int) -> Path:
    return RAW_RESULT_ROOT / dataset_key / variant_name / f'seed{seed}'


def completed_summary_to_row(summary: dict[str, Any]) -> dict[str, Any]:
    row = {
        'dataset': summary['dataset'],
        'variant': summary['variant'],
        'variant_display_name': summary.get('variant_display_name', summary['variant']),
        'seed': summary['seed'],
        'task_type': summary['task_type'],
        'inference_mode': summary.get('inference_mode', 'mean'),
        'primary_metric_name': summary['primary_metric_name'],
        'primary_metric_value': summary.get('test_primary_value', summary.get('primary_metric_value')),
        'primary_metric_signed': summary.get('test_primary_signed', summary.get('primary_metric_signed')),
        'train_primary_value': summary.get('train_primary_value'),
        'train_primary_signed': summary.get('train_primary_signed'),
        'val_primary_value': summary.get('val_primary_value'),
        'val_primary_signed': summary.get('val_primary_signed'),
        'test_primary_value': summary.get('test_primary_value', summary.get('primary_metric_value')),
        'test_primary_signed': summary.get('test_primary_signed', summary.get('primary_metric_signed')),
        'best_epoch': summary.get('best_epoch', 0),
        'train_time_sec': summary.get('train_time_sec', 0.0),
        'n_parameters': summary.get('n_parameters', 0),
        'git_commit': summary.get('git_commit', PINNED_TABM_COMMIT),
        'amp_enabled': summary.get('amp_enabled'),
        'amp_dtype': summary.get('amp_dtype'),
    }
    for part in ['train', 'val', 'test']:
        for key, value in summary.get('metrics', {}).get(part, {}).items():
            row[f'{part}_{key}'] = value
    for key, value in summary.get('module_diagnostics', {}).items():
        if isinstance(value, (int, float)):
            row[f'diagnostic_{key}'] = value
    return row


def report_to_row(report_payload: dict[str, Any]) -> dict[str, Any]:
    return completed_summary_to_row({
        'dataset': report_payload['dataset'],
        'variant': report_payload['variant'],
        'variant_display_name': report_payload['variant_display_name'],
        'seed': report_payload['seed'],
        'task_type': report_payload['task_type'],
        'inference_mode': report_payload['inference_mode'],
        'primary_metric_name': report_payload['score_name'],
        'train_primary_value': report_payload['metrics']['train'][metric_key_from_score_name(report_payload['score_name'])],
        'train_primary_signed': report_payload['metrics']['train']['score'],
        'val_primary_value': report_payload['metrics']['val'][metric_key_from_score_name(report_payload['score_name'])],
        'val_primary_signed': report_payload['metrics']['val']['score'],
        'test_primary_value': report_payload['metrics']['test'][metric_key_from_score_name(report_payload['score_name'])],
        'test_primary_signed': report_payload['metrics']['test']['score'],
        'best_epoch': report_payload['best_epoch'],
        'train_time_sec': report_payload['train_time_sec'],
        'n_parameters': report_payload['n_parameters'],
        'metrics': report_payload['metrics'],
        'module_diagnostics': report_payload.get('module_diagnostics', {}),
        'git_commit': report_payload.get('git_commit', PINNED_TABM_COMMIT),
        'amp_enabled': report_payload.get('amp_enabled'),
        'amp_dtype': report_payload.get('amp_dtype'),
    })


def generate_train_batches(
    *,
    n_train: int,
    batch_size: int,
    k: int,
    share_training_batches: bool,
    batch_generator: torch.Generator,
) -> list[torch.Tensor]:
    if share_training_batches:
        return list(torch.randperm(n_train, generator=batch_generator).split(batch_size))
    return [
        batch.transpose(0, 1).reshape(-1)
        for batch in torch.rand((k, n_train), generator=batch_generator).argsort(dim=1).split(batch_size, dim=1)
    ]


def fetch_batch(bundle: dict[str, Any], part: str, indices: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    tensors = bundle['tensors'][part]
    indices = indices.to(dtype=torch.long, device='cpu')
    x_num = tensors['x_num'][indices].to(device, non_blocking=True)
    x_cat = tensors['x_cat'][indices].to(device, non_blocking=True)
    y = tensors['y'][indices].to(device, non_blocking=True)
    return x_num, x_cat, y


def sample_shared_aux_batch(
    bundle: dict[str, Any],
    *,
    batch_generator: torch.Generator,
    batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    n_train = bundle['sizes']['train']
    actual = min(max(1, batch_size), n_train)
    idx = torch.randperm(n_train, generator=batch_generator)[:actual]
    return fetch_batch(bundle, 'train', idx)


def paper_memberwise_loss(
    logits: torch.Tensor,
    target: torch.Tensor,
    task_type: str,
    *,
    share_training_batches: bool,
) -> torch.Tensor:
    flat_logits = logits.flatten(0, 1)
    if task_type == 'binclass':
        flat_target = target.long().repeat_interleave(logits.shape[1]) if share_training_batches else target.long()
        return F.cross_entropy(flat_logits, flat_target)
    flat_target = target.float().repeat_interleave(logits.shape[1]) if share_training_batches else target.float()
    return F.mse_loss(flat_logits.reshape(-1), flat_target.reshape(-1))


def evaluate_model(model: nn.Module, loader: DataLoader, bundle: dict[str, Any]) -> dict[str, Any]:
    model.eval()
    y_true_batches = []
    head_logits_batches = []
    router_logits_batches = []
    with torch.inference_mode():
        for x_num, x_cat, y in loader:
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            y_true_batches.append(y.numpy())
            if hasattr(model, 'forward_components'):
                payload = model.forward_components(x_num, x_cat, force_shared_batches=True)
                head_logits = payload.get('final_member_logits', payload['specialist_member_logits']).float()
                router_logits = payload.get('final_ensemble_logits')
                if router_logits is not None:
                    router_logits_batches.append(router_logits.float().cpu())
            else:
                head_logits = model(x_num, x_cat, force_shared_batches=True).float()
            head_logits_batches.append(head_logits.cpu())

    y_true = np.concatenate(y_true_batches, axis=0)
    head_logits = torch.cat(head_logits_batches, dim=0)
    metric_key = metric_key_from_score_name(bundle['score_name'])

    if bundle['task_type'] == 'binclass':
        head_probs_full = torch.softmax(head_logits, dim=-1).cpu().numpy()
        head_predictions = head_probs_full[..., 1]
        mean_prediction = head_predictions.mean(axis=1)
        router_prediction = None
        if router_logits_batches:
            router_logits = torch.cat(router_logits_batches, dim=0)
            router_prediction = torch.softmax(router_logits, dim=-1).cpu().numpy()[..., 1]
        return {
            'y_true': y_true.astype(np.int64, copy=False),
            'score_name': bundle['score_name'],
            'metric_key': metric_key,
            'head_predictions': head_predictions,
            'head_probabilities': head_probs_full,
            'mean_prediction': mean_prediction,
            'router_prediction': router_prediction,
        }

    head_predictions = head_logits.squeeze(-1).cpu().numpy()
    if bundle['regression_stats'] is not None:
        stats = bundle['regression_stats']
        head_predictions = head_predictions * stats.std + stats.mean
    mean_prediction = head_predictions.mean(axis=1)
    router_prediction = None
    if router_logits_batches:
        router_prediction = torch.cat(router_logits_batches, dim=0).squeeze(-1).cpu().numpy()
        if bundle['regression_stats'] is not None:
            stats = bundle['regression_stats']
            router_prediction = router_prediction * stats.std + stats.mean
    return {
        'y_true': y_true.astype(np.float32, copy=False),
        'score_name': bundle['score_name'],
        'metric_key': metric_key,
        'head_predictions': head_predictions,
        'head_probabilities': None,
        'mean_prediction': mean_prediction,
        'router_prediction': router_prediction,
    }


def compute_metrics_from_predictions(
    bundle: dict[str, Any],
    eval_payload: dict[str, Any],
    prediction: np.ndarray,
) -> dict[str, float]:
    y_true = eval_payload['y_true']
    member_output = eval_payload['head_predictions']
    if bundle['task_type'] == 'binclass':
        probs = np.clip(prediction.reshape(-1), 1e-6, 1.0 - 1e-6)
        pred_labels = (probs >= 0.5).astype(np.int64)
        member_positive = np.clip(member_output, 1e-6, 1.0 - 1e-6)
        member_probs = np.stack([1.0 - member_positive, member_positive], axis=-1)
        member_labels = (member_positive >= 0.5).astype(np.int64)
        member_log_losses = [skm.log_loss(y_true, member_positive[:, i], labels=[0, 1]) for i in range(member_positive.shape[1])]
        member_accuracies = [skm.accuracy_score(y_true, member_labels[:, i]) for i in range(member_positive.shape[1])]
        return {
            'roc_auc': safe_roc_auc(y_true, probs),
            'accuracy': float(skm.accuracy_score(y_true, pred_labels)),
            'log_loss': float(skm.log_loss(y_true, probs, labels=[0, 1])),
            'score': signed_score(bundle['score_name'], safe_roc_auc(y_true, probs)),
            'mean_pairwise_prob_corr': mean_pairwise_corr(member_positive),
            'mean_pairwise_kl': mean_pairwise_binary_kl(member_probs),
            'mean_pairwise_disagreement': mean_pairwise_disagreement(member_labels),
            'mean_member_log_loss': float(np.mean(member_log_losses)),
            'mean_member_accuracy': float(np.mean(member_accuracies)),
            'ensemble_gain_log_loss': float(np.mean(member_log_losses) - skm.log_loss(y_true, probs, labels=[0, 1])),
            'ensemble_gain_accuracy': float(skm.accuracy_score(y_true, pred_labels) - np.mean(member_accuracies)),
        }
    preds = prediction.reshape(-1)
    residuals = member_output - y_true.reshape(-1, 1)
    member_rmses = [math.sqrt(skm.mean_squared_error(y_true, member_output[:, i])) for i in range(member_output.shape[1])]
    rmse_value = float(math.sqrt(skm.mean_squared_error(y_true, preds)))
    return {
        'rmse': rmse_value,
        'mae': float(skm.mean_absolute_error(y_true, preds)),
        'r2': float(skm.r2_score(y_true, preds)),
        'score': signed_score(bundle['score_name'], rmse_value),
        'mean_pairwise_prediction_corr': mean_pairwise_corr(member_output),
        'mean_pairwise_residual_corr': mean_pairwise_corr(residuals),
        'member_variance': float(np.var(member_output, axis=1).mean()),
        'mean_member_rmse': float(np.mean(member_rmses)),
        'ensemble_gain_rmse': float(np.mean(member_rmses) - rmse_value),
    }


def score_single_prediction(bundle: dict[str, Any], y_true: np.ndarray, prediction: np.ndarray) -> float:
    if bundle['score_name'] == 'roc-auc':
        return safe_roc_auc(y_true, prediction.reshape(-1))
    return float(math.sqrt(skm.mean_squared_error(y_true, prediction.reshape(-1))))


def select_head_sets(bundle: dict[str, Any], val_eval: dict[str, Any]) -> dict[str, Any]:
    head_predictions = val_eval['head_predictions']
    y_true = val_eval['y_true']
    n_heads = head_predictions.shape[1]
    raw_scores = np.array([score_single_prediction(bundle, y_true, head_predictions[:, i]) for i in range(n_heads)])
    signed_scores = np.array([signed_score(bundle['score_name'], float(x)) for x in raw_scores])
    best_head_idx = int(np.nanargmax(signed_scores))
    greedy_heads = [best_head_idx]
    greedy_signed = signed_scores[best_head_idx]
    available = [True] * n_heads
    available[best_head_idx] = False
    while len(greedy_heads) < n_heads:
        new_idx = None
        new_signed = None
        for head_idx in range(n_heads):
            if not available[head_idx]:
                continue
            candidate_heads = [*greedy_heads, head_idx]
            candidate_pred = head_predictions[:, candidate_heads].mean(axis=1)
            candidate_raw = score_single_prediction(bundle, y_true, candidate_pred)
            candidate_signed = signed_score(bundle['score_name'], candidate_raw)
            if candidate_signed > greedy_signed and (new_signed is None or candidate_signed > new_signed):
                new_idx = head_idx
                new_signed = candidate_signed
        if new_idx is None:
            break
        greedy_heads.append(new_idx)
        available[new_idx] = False
        greedy_signed = float(new_signed)
    return {
        'best_head_idx': best_head_idx,
        'greedy_heads': greedy_heads,
        'head_val_scores': raw_scores.tolist(),
    }


def collect_ramp_stats(model: RampResidualTabM, loader: DataLoader) -> dict[str, float]:
    gate_snapshot = model.gates.snapshot()
    stats = {
        'gate_mean': float(gate_snapshot['feature_gates'].mean()),
        'gate_std': float(gate_snapshot['feature_gates'].std()),
        'gate_min': float(gate_snapshot['feature_gates'].min()),
        'gate_max': float(gate_snapshot['feature_gates'].max()),
    }
    if model.router is None:
        stats['router_entropy'] = float('nan')
        stats['router_load_balance'] = float('nan')
        return stats
    weights = []
    with torch.inference_mode():
        for x_num, x_cat, _ in loader:
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            encoded = model.encode_inputs(x_num, x_cat)
            weights.append(model.router(encoded).detach().cpu())
    if not weights:
        stats['router_entropy'] = float('nan')
        stats['router_load_balance'] = float('nan')
        return stats
    router_weights = torch.cat(weights, dim=0)
    stats['router_entropy'] = float(route_entropy(router_weights).cpu().item())
    stats['router_load_balance'] = float(route_balance_loss(router_weights).cpu().item())
    return stats


def existing_run_row(bundle: dict[str, Any], variant_name: str, seed: int) -> dict[str, Any] | None:
    report_path = run_dir_for(bundle['dataset_key'], variant_name, seed, 'mean') / 'report.json'
    done_path = run_dir_for(bundle['dataset_key'], variant_name, seed, 'mean') / 'DONE'
    if report_path.exists() and done_path.exists() and not FORCE_RERUN:
        return report_to_row(json.loads(report_path.read_text(encoding='utf-8')))
    return None


def train_variant(bundle: dict[str, Any], variant_name: str, seed: int) -> dict[str, Any]:
    existing = existing_run_row(bundle, variant_name, seed)
    if existing is not None:
        existing['source_kind'] = 'current'
        return existing

    variant_cfg = resolve_variant_config(variant_name, bundle['dataset_key'])
    config_payload = base_run_config(bundle, variant_name, seed)
    share_training_batches = bool(config_payload['model']['share_training_batches'])
    mean_run_dir = run_dir_for(bundle['dataset_key'], variant_name, seed, 'mean')
    mean_run_dir.mkdir(parents=True, exist_ok=True)
    write_toml(mean_run_dir / '0.toml', config_payload)

    set_seed(seed)
    model = build_model(bundle, variant_name, seed)
    if variant_cfg.get('use_baseline_init'):
        checkpoint_path = baseline_checkpoint_path(bundle, seed)
        if checkpoint_path is None or not checkpoint_path.exists():
            raise FileNotFoundError(f'Baseline checkpoint required for baseline-initialized variant: {checkpoint_path}')
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        load_matching_state(model, checkpoint['model_state_dict'])

    optimizer = torch.optim.AdamW(
        make_parameter_groups(model),
        lr=config_payload['optimizer']['lr'],
        weight_decay=config_payload['optimizer']['weight_decay'],
    )
    amp_requested = bool(config_payload['amp']) and USE_AMP
    amp_dtype = torch.bfloat16 if amp_requested and device.type == 'cuda' and torch.cuda.is_bf16_supported() else None
    amp_enabled = amp_dtype is not None
    loaders = make_tensor_dataloaders(bundle, batch_size=config_payload['batch_size'], eval_batch_size=EVAL_BATCH_SIZE)
    best_state = None
    best_val_signed = -float('inf')
    best_epoch = 0
    patience_left = int(config_payload['patience'])
    history_rows = []
    train_started_at = time.time()
    progress_path = ARTIFACT_ROOT / 'current_progress.json'
    batch_generator = torch.Generator(device='cpu').manual_seed(seed)
    total_steps = 0
    epoch_size = int(math.ceil(bundle['sizes']['train'] / config_payload['batch_size']))

    try:
        for epoch in range(1, int(config_payload['n_epochs']) + 1):
            if hasattr(model, 'set_epoch'):
                model.set_epoch(epoch)
            model.train()
            epoch_losses = []
            epoch_task_losses = []
            epoch_diversities = []
            epoch_gate_losses = []
            epoch_route_losses = []
            epoch_fidelity_losses = []
            batches = generate_train_batches(
                n_train=bundle['sizes']['train'],
                batch_size=config_payload['batch_size'],
                k=32,
                share_training_batches=share_training_batches,
                batch_generator=batch_generator,
            )
            for batch_idx in tqdm(batches, desc=f"{bundle['dataset_key']} | {variant_name} | seed {seed} | epoch {epoch}", leave=False):
                total_steps += 1
                x_num, x_cat, y = fetch_batch(bundle, 'train', batch_idx)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
                    if variant_cfg['family'] == 'legacy':
                        logits = model(x_num, x_cat, force_shared_batches=share_training_batches)
                        task_loss = paper_memberwise_loss(
                            logits,
                            y,
                            bundle['task_type'],
                            share_training_batches=share_training_batches,
                        )
                        loss = task_loss
                        diversity_value = torch.zeros((), device=device)
                        gate_loss = torch.zeros((), device=device)
                        route_loss = torch.zeros((), device=device)
                        fidelity_term = torch.zeros((), device=device)
                    else:
                        payload = model.forward_components(
                            x_num,
                            x_cat,
                            force_shared_batches=share_training_batches,
                        )
                        task_loss = paper_memberwise_loss(
                            payload['final_member_logits'],
                            y,
                            bundle['task_type'],
                            share_training_batches=share_training_batches,
                        )
                        gate_budget_loss = model.gates.budget_loss(variant_cfg['budget_target'])
                        gate_div_loss = model.gates.diversity_loss()
                        diversity_value = torch.zeros((), device=device)
                        route_loss = torch.zeros((), device=device)
                        fidelity_term = torch.zeros((), device=device)
                        if total_steps % max(1, int(variant_cfg['aux_shared_interval'])) == 0 and int(variant_cfg['aux_shared_batch_size']) > 0:
                            aux_x_num, aux_x_cat, _ = sample_shared_aux_batch(
                                bundle,
                                batch_generator=batch_generator,
                                batch_size=int(variant_cfg['aux_shared_batch_size']),
                            )
                            aux_payload = model.forward_components(aux_x_num, aux_x_cat, force_shared_batches=True)
                            diversity_value = ncl_diversity(aux_payload['specialist_member_logits'], bundle['task_type'])
                            if aux_payload['router_weights'] is not None:
                                route_loss = route_balance_loss(aux_payload['router_weights']) - 0.05 * route_entropy(aux_payload['router_weights'])
                            if variant_cfg['lambda_fidelity'] > 0:
                                fidelity_progress = min(1.0, epoch / max(1.0, int(config_payload['n_epochs']) * variant_cfg['fidelity_warmup_fraction']))
                                fidelity_weight = variant_cfg['lambda_fidelity'] * max(0.0, 1.0 - fidelity_progress)
                                if fidelity_weight > 0:
                                    fidelity_term = fidelity_weight * fidelity_loss(
                                        aux_payload['specialist_member_logits'].mean(dim=1),
                                        aux_payload['baseline_member_logits'].mean(dim=1),
                                        bundle['task_type'],
                                    )
                        gate_loss = gate_budget_loss + gate_div_loss
                        loss = (
                            task_loss
                            + variant_cfg['lambda_budget'] * gate_budget_loss
                            + variant_cfg['lambda_gate_div'] * gate_div_loss
                            + variant_cfg['lambda_route'] * route_loss
                            + fidelity_term
                            - variant_cfg['lambda_ncl'] * diversity_value
                        )
                if not bool(torch.isfinite(loss).item()):
                    failure_payload = {
                        'dataset': bundle['dataset_key'],
                        'variant': variant_name,
                        'variant_display_name': variant_cfg['display_name'],
                        'seed': seed,
                        'task_type': bundle['task_type'],
                        'score_name': bundle['score_name'],
                        'inference_mode': 'mean',
                        'config': config_payload,
                        'n_parameters': get_n_parameters(model),
                        'prediction_type': prediction_type_for_task(bundle['task_type']),
                        'epoch_size': epoch_size,
                        'metrics': {},
                        'time': format_duration(time.time() - train_started_at),
                        'best_epoch': best_epoch,
                        'train_time_sec': float(time.time() - train_started_at),
                        'amp_enabled': amp_enabled,
                        'amp_dtype': amp_dtype_string(amp_dtype),
                        'git_commit': PINNED_TABM_COMMIT,
                        'failure': {
                            'reason': 'non_finite_loss',
                            'epoch': epoch,
                            'step': total_steps,
                            'amp_dtype': amp_dtype_string(amp_dtype),
                        },
                    }
                    write_json(mean_run_dir / 'report.json', failure_payload)
                    raise AssertionError(f'Non-finite loss at epoch={epoch} step={total_steps}')
                loss.backward()
                if config_payload['gradient_clipping_norm'] > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config_payload['gradient_clipping_norm'])
                optimizer.step()
                epoch_losses.append(float(loss.detach().cpu().item()))
                epoch_task_losses.append(float(task_loss.detach().cpu().item()))
                epoch_diversities.append(float(diversity_value.detach().cpu().item()))
                epoch_gate_losses.append(float(gate_loss.detach().cpu().item()))
                epoch_route_losses.append(float(route_loss.detach().cpu().item()))
                epoch_fidelity_losses.append(float(fidelity_term.detach().cpu().item()))

            val_eval = evaluate_model(model, loaders['val'], bundle)
            val_metrics = compute_metrics_from_predictions(bundle, val_eval, val_eval['mean_prediction'])
            history_rows.append({
                'epoch': epoch,
                'train_loss': float(np.mean(epoch_losses)),
                'train_task_loss': float(np.mean(epoch_task_losses)),
                'train_diversity': float(np.mean(epoch_diversities)),
                'train_gate_loss': float(np.mean(epoch_gate_losses)),
                'train_route_loss': float(np.mean(epoch_route_losses)),
                'train_fidelity_loss': float(np.mean(epoch_fidelity_losses)),
                'val_primary_value': val_metrics[val_eval['metric_key']],
                'val_primary_signed': val_metrics['score'],
            })
            write_json(progress_path, {
                'dataset': bundle['dataset_key'],
                'variant': variant_name,
                'seed': seed,
                'epoch': epoch,
                'best_epoch': best_epoch,
                'best_val_signed': best_val_signed,
            })
            if val_metrics['score'] > best_val_signed:
                best_val_signed = float(val_metrics['score'])
                best_state = state_dict_to_cpu(model)
                best_epoch = epoch
                patience_left = int(config_payload['patience'])
                torch.save(
                    {
                        'model_state_dict': best_state,
                        'epoch': epoch,
                        'val_primary_signed': best_val_signed,
                    },
                    mean_run_dir / 'checkpoint_best.pt',
                )
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break
    except Exception:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise

    if best_state is None:
        raise RuntimeError(f'No best checkpoint was captured for {bundle["dataset_key"]} / {variant_name} / seed {seed}')

    model.load_state_dict(best_state)
    train_eval = evaluate_model(model, loaders['train'], bundle)
    val_eval = evaluate_model(model, loaders['val'], bundle)
    test_eval = evaluate_model(model, loaders['test'], bundle)
    train_time_sec = float(time.time() - train_started_at)
    n_parameters = int(get_n_parameters(model))
    selection = select_head_sets(bundle, val_eval)

    if variant_cfg['family'] == 'legacy':
        mask_snapshot = model.current_mask_snapshot()
        module_diagnostics = dict(model.mask_stats)
        module_diagnostics['mask_mode'] = variant_cfg['mask_mode']
        module_diagnostics['mask_granularity'] = variant_cfg['mask_granularity']
        module_diagnostics['configured_keep_rate'] = variant_cfg['keep_rate']
        module_diagnostics['configured_mask_strength'] = variant_cfg['mask_strength']
        module_diagnostics['configured_anchor_fraction'] = variant_cfg['anchor_fraction']
        module_diagnostics['configured_core_fraction'] = variant_cfg['core_fraction']
        module_diagnostics['configured_warmup_epochs'] = variant_cfg['warmup_epochs']
        if mask_snapshot['feature_mask'] is not None:
            np.save(mean_run_dir / 'feature_mask.npy', mask_snapshot['feature_mask'])
        if mask_snapshot['dim_mask'] is not None:
            np.save(mean_run_dir / 'dim_mask.npy', mask_snapshot['dim_mask'])
    else:
        gate_snapshot = model.gates.snapshot()
        save_array_bundle(mean_run_dir / 'learned_gates.npz', gate_snapshot)
        module_diagnostics = {
            'family': 'ramp',
            'configured_budget_target': float(variant_cfg['budget_target']),
            'configured_core_floor': float(variant_cfg['core_floor']),
            'configured_lambda_ncl': float(variant_cfg['lambda_ncl']),
            'configured_lambda_budget': float(variant_cfg['lambda_budget']),
            'configured_lambda_gate_div': float(variant_cfg['lambda_gate_div']),
            'configured_lambda_route': float(variant_cfg['lambda_route']),
            'configured_lambda_fidelity': float(variant_cfg['lambda_fidelity']),
            'configured_residual_rho': float(variant_cfg['residual_rho']),
            'configured_use_router': float(1.0 if variant_cfg['use_router'] else 0.0),
            **collect_ramp_stats(model, loaders['val']),
        }
    write_json(mean_run_dir / 'module_diagnostics.json', module_diagnostics)

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(mean_run_dir / 'history.csv', index=False)
    save_array_bundle(mean_run_dir / 'predictions_val.npz', {
        'y_true': val_eval['y_true'],
        'head_predictions': val_eval['head_predictions'],
        'mean_prediction': val_eval['mean_prediction'],
        'router_prediction': np.array([]) if val_eval['router_prediction'] is None else val_eval['router_prediction'],
    })
    save_array_bundle(mean_run_dir / 'predictions_test.npz', {
        'y_true': test_eval['y_true'],
        'head_predictions': test_eval['head_predictions'],
        'mean_prediction': test_eval['mean_prediction'],
        'router_prediction': np.array([]) if test_eval['router_prediction'] is None else test_eval['router_prediction'],
    })

    def mode_prediction(eval_payload: dict[str, Any], inference_mode: str) -> np.ndarray:
        if inference_mode == 'mean':
            return eval_payload['mean_prediction']
        if inference_mode == 'best-head':
            return eval_payload['head_predictions'][:, selection['best_head_idx']]
        if inference_mode == 'greedy-heads':
            return eval_payload['head_predictions'][:, selection['greedy_heads']].mean(axis=1)
        if inference_mode == 'router':
            if eval_payload['router_prediction'] is None:
                raise KeyError('Router prediction is not available')
            return eval_payload['router_prediction']
        raise KeyError(inference_mode)

    common_report = {
        'dataset': bundle['dataset_key'],
        'variant': variant_name,
        'variant_display_name': variant_cfg['display_name'],
        'seed': seed,
        'task_type': bundle['task_type'],
        'score_name': bundle['score_name'],
        'config': config_payload,
        'n_parameters': n_parameters,
        'prediction_type': prediction_type_for_task(bundle['task_type']),
        'epoch_size': epoch_size,
        'time': format_duration(train_time_sec),
        'best_epoch': best_epoch,
        'train_time_sec': train_time_sec,
        'amp_enabled': amp_enabled,
        'amp_dtype': amp_dtype_string(amp_dtype),
        'git_commit': PINNED_TABM_COMMIT,
        'module_diagnostics': module_diagnostics,
        'head_selection': selection,
    }

    mode_rows = {}
    for inference_mode in ['mean', 'best-head', 'greedy-heads']:
        metrics = {
            'train': compute_metrics_from_predictions(bundle, train_eval, mode_prediction(train_eval, inference_mode)),
            'val': compute_metrics_from_predictions(bundle, val_eval, mode_prediction(val_eval, inference_mode)),
            'test': compute_metrics_from_predictions(bundle, test_eval, mode_prediction(test_eval, inference_mode)),
        }
        report_payload = common_report | {
            'inference_mode': inference_mode,
            'metrics': metrics,
        }
        finish_run_dir(
            bundle=bundle,
            variant_name=variant_name,
            seed=seed,
            inference_mode=inference_mode,
            config_payload=config_payload | {'inference_mode': inference_mode},
            report_payload=report_payload,
        )
        mode_rows[inference_mode] = report_to_row(report_payload)
    if test_eval['router_prediction'] is not None:
        router_metrics = {
            'train': compute_metrics_from_predictions(bundle, train_eval, mode_prediction(train_eval, 'router')),
            'val': compute_metrics_from_predictions(bundle, val_eval, mode_prediction(val_eval, 'router')),
            'test': compute_metrics_from_predictions(bundle, test_eval, mode_prediction(test_eval, 'router')),
        }
        router_report = common_report | {
            'inference_mode': 'router',
            'metrics': router_metrics,
        }
        finish_run_dir(
            bundle=bundle,
            variant_name=variant_name,
            seed=seed,
            inference_mode='router',
            config_payload=config_payload | {'inference_mode': 'router'},
            report_payload=router_report,
        )

    raw_dir = raw_run_dir(bundle['dataset_key'], variant_name, seed)
    raw_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(mean_run_dir / 'checkpoint_best.pt', raw_dir / 'checkpoint_best.pt')
    shutil.copy2(mean_run_dir / 'history.csv', raw_dir / 'history.csv')
    shutil.copy2(mean_run_dir / 'module_diagnostics.json', raw_dir / 'mask_stats.json')
    shutil.copy2(mean_run_dir / 'predictions_val.npz', raw_dir / 'predictions_val.npz')
    shutil.copy2(mean_run_dir / 'predictions_test.npz', raw_dir / 'predictions_test.npz')
    raw_summary = {
        'dataset': bundle['dataset_key'],
        'variant': variant_name,
        'variant_display_name': variant_cfg['display_name'],
        'seed': seed,
        'task_type': bundle['task_type'],
        'primary_metric_name': bundle['score_name'],
        'primary_metric_value': mode_rows['mean']['test_primary_value'],
        'primary_metric_signed': mode_rows['mean']['test_primary_signed'],
        'train_primary_value': mode_rows['mean']['train_primary_value'],
        'train_primary_signed': mode_rows['mean']['train_primary_signed'],
        'val_primary_value': mode_rows['mean']['val_primary_value'],
        'val_primary_signed': mode_rows['mean']['val_primary_signed'],
        'test_primary_value': mode_rows['mean']['test_primary_value'],
        'test_primary_signed': mode_rows['mean']['test_primary_signed'],
        'best_epoch': best_epoch,
        'train_time_sec': train_time_sec,
        'n_parameters': n_parameters,
        'train_metrics': common_report | {'metrics': {'train': compute_metrics_from_predictions(bundle, train_eval, train_eval['mean_prediction'])}},
        'val_metrics': common_report | {'metrics': {'val': compute_metrics_from_predictions(bundle, val_eval, val_eval['mean_prediction'])}},
        'test_metrics': common_report | {'metrics': {'test': compute_metrics_from_predictions(bundle, test_eval, test_eval['mean_prediction'])}},
        'mask_stats': module_diagnostics,
        'variant_config': variant_cfg,
        'inference_mode': 'mean',
        'git_commit': PINNED_TABM_COMMIT,
        'amp_enabled': amp_enabled,
        'amp_dtype': amp_dtype_string(amp_dtype),
    }
    raw_summary['train_metrics'] = compute_metrics_from_predictions(bundle, train_eval, train_eval['mean_prediction'])
    raw_summary['val_metrics'] = compute_metrics_from_predictions(bundle, val_eval, val_eval['mean_prediction'])
    raw_summary['test_metrics'] = compute_metrics_from_predictions(bundle, test_eval, test_eval['mean_prediction'])
    write_json(raw_dir / 'completed.json', raw_summary)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    result = dict(mode_rows['mean'])
    result['source_kind'] = 'current'
    return result


def infer_inference_mode(dir_name: str) -> str:
    if dir_name.endswith('-best-head-evaluation'):
        return 'best-head'
    if dir_name.endswith('-greedy-heads-evaluation'):
        return 'greedy-heads'
    if dir_name.endswith('-router-evaluation'):
        return 'router'
    if dir_name.endswith('-evaluation'):
        return 'mean'
    raise ValueError(f'Unknown evaluation directory name: {dir_name}')


def aggregate_team_reports() -> dict[str, pd.DataFrame]:
    rows = []
    for report_path in TEAM_EXPERIMENT_ROOT.rglob('report.json'):
        run_dir = report_path.parent
        if not run_dir.joinpath('DONE').exists():
            continue
        dir_name = run_dir.parent.name
        inference_mode = infer_inference_mode(dir_name)
        report = json.loads(report_path.read_text(encoding='utf-8'))
        config_path = run_dir / '0.toml'
        if not config_path.exists():
            continue
        config_toml = tomllib.loads(config_path.read_text(encoding='utf-8'))
        if report.get('config', {}).get('amp') != config_toml.get('amp'):
            continue
        if int(report.get('seed', -1)) != int(run_dir.name):
            continue
        row = report_to_row(report)
        row['inference_mode'] = inference_mode
        row['report_path'] = str(report_path)
        row['config_path'] = str(config_path)
        rows.append(row)
    if not rows:
        raise RuntimeError('No completed team-format reports were found.')
    long_df = pd.DataFrame(rows).sort_values(['dataset', 'variant', 'inference_mode', 'seed']).reset_index(drop=True)
    TEAM_AGGREGATED_ROOT.mkdir(parents=True, exist_ok=True)
    long_df.to_csv(TEAM_AGGREGATED_ROOT / 'long_results.csv', index=False)
    long_df.to_csv(TABLE_ROOT / 'current_all_results.csv', index=False)
    long_df.to_csv(TABLE_ROOT / 'final_all_results.csv', index=False)

    metric_columns = [
        'train_primary_value',
        'val_primary_value',
        'val_primary_signed',
        'test_primary_value',
        'test_primary_signed',
        'test_accuracy',
        'test_log_loss',
        'test_roc_auc',
        'test_rmse',
        'test_mae',
        'test_r2',
        'train_time_sec',
        'n_parameters',
    ]
    metric_columns += [x for x in long_df.columns if x.startswith('diagnostic_')]
    metric_columns = [x for x in metric_columns if x in long_df.columns]
    grouped = long_df.groupby(
        ['dataset', 'variant', 'variant_display_name', 'task_type', 'inference_mode', 'primary_metric_name'],
        as_index=False,
    )[metric_columns]
    mean_df = grouped.mean()
    std_df = grouped.std(ddof=0)
    wide_df = mean_df.copy()
    for column in metric_columns:
        wide_df[f'{column}_std'] = std_df[column]
    wide_df.to_csv(TEAM_AGGREGATED_ROOT / 'wide_summary.csv', index=False)
    wide_df.to_csv(TABLE_ROOT / 'final_mean_std.csv', index=False)

    selection_rows = []
    comparison_rows = []
    for dataset_key in DATASET_SEQUENCE:
        subset = wide_df[wide_df['dataset'] == dataset_key].copy()
        if subset.empty:
            continue
        subset = subset.sort_values(['val_primary_signed', 'variant', 'inference_mode'], ascending=[False, True, True]).reset_index(drop=True)
        baseline_subset = subset[subset['variant'] == 'tabm_baseline']
        candidate_subset = subset[subset['variant'] != 'tabm_baseline']
        if baseline_subset.empty or candidate_subset.empty:
            continue
        baseline_row = baseline_subset.iloc[0]
        selected_row = candidate_subset.iloc[0]
        selection_rows.append({
            'dataset': dataset_key,
            'selected_variant': selected_row['variant'],
            'selected_variant_display_name': selected_row['variant_display_name'],
            'selected_inference_mode': selected_row['inference_mode'],
            'selection_metric_name': selected_row['primary_metric_name'],
            'selection_val_primary_value': selected_row['val_primary_value'],
            'selection_val_primary_signed': selected_row['val_primary_signed'],
            'selection_n_seeds': int(long_df[(long_df['dataset'] == dataset_key) & (long_df['variant'] == selected_row['variant']) & (long_df['inference_mode'] == selected_row['inference_mode'])]['seed'].nunique()),
        })
        comparison_rows.append({
            'dataset': dataset_key,
            'metric_name': selected_row['primary_metric_name'],
            'baseline_variant': baseline_row['variant'],
            'baseline_inference_mode': baseline_row['inference_mode'],
            'baseline_mean': baseline_row['test_primary_value'],
            'baseline_std': baseline_row['test_primary_value_std'],
            'selected_variant': selected_row['variant'],
            'selected_inference_mode': selected_row['inference_mode'],
            'selected_mean': selected_row['test_primary_value'],
            'selected_std': selected_row['test_primary_value_std'],
            'improvement_pct': relative_improvement(selected_row['primary_metric_name'], baseline_row['test_primary_value'], selected_row['test_primary_value']),
            'is_positive_improvement': bool(relative_improvement(selected_row['primary_metric_name'], baseline_row['test_primary_value'], selected_row['test_primary_value']) > 0.0),
        })
    selected_df = pd.DataFrame(selection_rows)
    comparison_df = pd.DataFrame(comparison_rows)
    selected_df.to_csv(TABLE_ROOT / 'screen_selected_variants.csv', index=False)
    comparison_df.to_csv(TABLE_ROOT / 'tc5_comparison.csv', index=False)
    comparison_df.to_csv(TABLE_ROOT / 'final_comparison.csv', index=False)
    pd.DataFrame([{'note': 'No validation blend in the compliant rerun; file kept only for backward compatibility.'}]).to_csv(
        TABLE_ROOT / 'blend_weights.csv',
        index=False,
    )

    summary_lines = ['# RAMP++-NCL-MFB TC5 Summary', '']
    summary_lines.append(f'- git commit: `{PINNED_TABM_COMMIT}`')
    summary_lines.append(f'- datasets: {", ".join(DATASET_SEQUENCE)}')
    summary_lines.append(f'- seeds: {FINAL_SEEDS}')
    summary_lines.append('')
    summary_lines.append('## Validation-Selected Variants')
    for _, row in selected_df.sort_values('dataset').iterrows():
        summary_lines.append(
            f"- {row['dataset']}: `{row['selected_variant']}` via `{row['selected_inference_mode']}` "
            f"(val {row['selection_metric_name']}={row['selection_val_primary_value']:.6f}, n_seeds={row['selection_n_seeds']})"
        )
    summary_lines.append('')
    summary_lines.append('## Baseline Comparison')
    for _, row in comparison_df.sort_values('dataset').iterrows():
        summary_lines.append(
            f"- {row['dataset']}: baseline `{row['baseline_variant']}`/{row['baseline_inference_mode']}={row['baseline_mean']:.6f}, "
            f"selected `{row['selected_variant']}`/{row['selected_inference_mode']}={row['selected_mean']:.6f}, "
            f"improvement={row['improvement_pct']:+.3f}%"
        )
    summary_text = '\n'.join(summary_lines)
    (TEAM_AGGREGATED_ROOT / 'summary.md').write_text(summary_text, encoding='utf-8')
    (TABLE_ROOT / 'summary.md').write_text(summary_text, encoding='utf-8')
    return {
        'long': long_df,
        'wide': wide_df,
        'selected': selected_df,
        'comparison': comparison_df,
    }


TEAM_RUN_VARIANT_MAP = {
    dataset_key: ['tabm_baseline', *CURRENT_SCREEN_VARIANT_MAP[dataset_key]]
    for dataset_key in DATASET_SEQUENCE
}

for dataset_key in DATASET_SEQUENCE:
    print('=' * 120)
    print(f'Compliant TC5 run | dataset={dataset_key} | seeds={FINAL_SEEDS} | split={SPLIT_NAME}')
    for seed in FINAL_SEEDS:
        bundle = prepare_dataset_bundle(dataset_key, SPLIT_NAME, seed)
        write_json(
            CACHE_ROOT / f'{dataset_key}_seed{seed}_bundle_summary.json',
            {
                'dataset': dataset_key,
                'seed': seed,
                'prepared_dir': bundle['prepared_dir'],
                'manifest': bundle['manifest'],
                'sizes': bundle['sizes'],
                'n_num_features': bundle['n_num_features'],
                'n_cat_features': len(bundle['cat_cardinalities']),
                'task_type': bundle['task_type'],
                'score_name': bundle['score_name'],
            },
        )
        for variant_name in TEAM_RUN_VARIANT_MAP[dataset_key]:
            row = train_variant(bundle, variant_name, seed)
            print(
                f"Completed {dataset_key} | {variant_name} | seed {seed} | "
                f"val={row['val_primary_value']:.6f} | test={row['test_primary_value']:.6f} | mode={row['inference_mode']}"
            )

artifacts = aggregate_team_reports()
display(artifacts['selected'].sort_values('dataset'))
display(artifacts['comparison'].sort_values('dataset'))
display(artifacts['long'][['dataset', 'variant', 'inference_mode', 'seed', 'val_primary_value', 'test_primary_value']].sort_values(['dataset', 'variant', 'inference_mode', 'seed']))
print('Finished compliant end-to-end RAMP++-NCL-MFB TC5 run.')
